In [1]:
#!/usr/bin/env python3
"""
LegalKGent — Transportation Law Data Downloader
================================================
Downloads UK transportation law data from multiple structured sources:
  1. Primary Legislation (Acts) from legislation.gov.uk
  2. Statutory Instruments from legislation.gov.uk  
  3. Case Law from National Archives
  4. Amendments Table (which Acts amend which) from legislation.gov.uk
  5. Explanatory Notes metadata

Focused on transportation domain to keep dataset manageable (~2000–3000 chunks).

Usage:
    python download_transport_data.py
"""

import os
import json
import time
import requests
from datetime import datetime

# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = "data"
LEGISLATION_DIR = os.path.join(OUTPUT_DIR, "raw_legislation")
CASELAW_DIR = os.path.join(OUTPUT_DIR, "raw_caselaw")
SI_DIR = os.path.join(OUTPUT_DIR, "raw_statutory_instruments")
AMENDMENTS_DIR = os.path.join(OUTPUT_DIR, "amendments")
NOTES_DIR = os.path.join(OUTPUT_DIR, "explanatory_notes")

for d in [LEGISLATION_DIR, CASELAW_DIR, SI_DIR, AMENDMENTS_DIR, NOTES_DIR]:
    os.makedirs(d, exist_ok=True)

HEADERS = {
    'User-Agent': 'LegalKGent-Research-Project/1.0 (university-research)',
    'Accept': 'application/xml, text/xml, */*'
}

POLITE_DELAY = 1.0  # seconds between requests

# ============================================================
# TRANSPORTATION LAW — CURATED ACT LIST
# ============================================================
# These are the key Acts in UK transportation law.
# We download a focused set rather than all Acts in a year.

TRANSPORT_ACTS = [
    # === Core Modern Transport Acts ===
    ("ukpga", 2024, 3,  "Automated Vehicles Act 2024"),
    ("ukpga", 2024, 2,  "Pedicabs (London) Act 2024"),
    
    # === Road Traffic ===
    ("ukpga", 1988, 52, "Road Traffic Act 1988"),
    ("ukpga", 1988, 53, "Road Traffic Offenders Act 1988"),
    ("ukpga", 1984, 27, "Road Traffic Regulation Act 1984"),
    ("ukpga", 2006, 49, "Road Safety Act 2006"),
    
    # === Transport & Railways ===
    ("ukpga", 2000, 38, "Transport Act 2000"),
    ("ukpga", 1993, 43, "Railways Act 1993"),
    ("ukpga", 2005, 14, "Railways Act 2005"),
    ("ukpga", 2008, 26, "Transport (London) Act 2008"),
    
    # === Aviation & Maritime ===
    ("ukpga", 2021, 12, "Air Traffic Management and Unmanned Aircraft Act 2021"),
    ("ukpga", 2012, 19, "Civil Aviation Act 2012"),
    
    # === Electric & Automated Vehicles ===
    ("ukpga", 2018, 18, "Automated and Electric Vehicles Act 2018"),
    
    # === Key Acts that AMEND transport legislation ===
    ("ukpga", 2022, 14, "Taxis and Private Hire Vehicles Act 2022"),
    ("ukpga", 1980, 34, "Highways Act 1980"),
    
    # === Recent Acts with transport implications ===
    ("ukpga", 2023, 54, "Online Safety Act 2023"),      # Regulates automated content delivery
    ("ukpga", 2023, 32, "Energy Act 2023"),              # EV charging infrastructure
]

# === Statutory Instruments (Transport SIs) ===
TRANSPORT_SIS = [
    # Key 2024 transport SIs
    ("uksi", 2024, 566, "Goods Vehicles (International Road Transport Permits) Regs 2024"),
    ("uksi", 2024, 615, "Motor Vehicles (Driving Licences) (Amendment) Regs 2024"),
    ("uksi", 2024, 305, "Road Vehicles (Registration and Licensing) (Amendment) Regs 2024"),
    
    # Key 2023 transport SIs
    ("uksi", 2023, 980, "Traffic Signs (Amendment) Regulations 2023"),
    ("uksi", 2023, 695, "Drivers' Hours and Tachographs (Amendment) Regs 2023"),
    ("uksi", 2023, 903, "Railways (Access, Management and Licensing) (Amendment) Regs 2023"),
    
    # Automated vehicles related
    ("uksi", 2022, 470, "Highway Code (Hierarchy of Road Users) Regs 2022"),
]


# ============================================================
# 1. DOWNLOAD PRIMARY LEGISLATION
# ============================================================

def download_legislation(acts_list, output_dir):
    """Download Acts from legislation.gov.uk as XML."""
    print(f"\n{'='*60}")
    print(f"📜 DOWNLOADING PRIMARY LEGISLATION")
    print(f"{'='*60}")
    
    success = 0
    for act_type, year, number, title in acts_list:
        filename = f"{act_type}_{year}_{number}.xml"
        filepath = os.path.join(output_dir, filename)
        
        if os.path.exists(filepath):
            print(f"  [SKIP] {filename} (already exists)")
            success += 1
            continue
        
        url = f"https://www.legislation.gov.uk/{act_type}/{year}/{number}/data.xml"
        
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            if r.status_code == 200 and r.content.strip().startswith(b"<"):
                with open(filepath, "wb") as f:
                    f.write(r.content)
                size_kb = len(r.content) / 1024
                print(f"  [OK] {title} ({size_kb:.0f} KB)")
                success += 1
            elif r.status_code == 404:
                print(f"  [404] {title} — not found at {url}")
            else:
                print(f"  [ERR] HTTP {r.status_code} for {title}")
        except Exception as e:
            print(f"  [ERR] {title}: {e}")
        
        time.sleep(POLITE_DELAY)
    
    print(f"\n  ✅ Downloaded {success}/{len(acts_list)} Acts")
    return success


# ============================================================
# 2. DOWNLOAD STATUTORY INSTRUMENTS
# ============================================================

def download_statutory_instruments(si_list, output_dir):
    """Download SIs from legislation.gov.uk as XML."""
    print(f"\n{'='*60}")
    print(f"📋 DOWNLOADING STATUTORY INSTRUMENTS")
    print(f"{'='*60}")
    
    success = 0
    for si_type, year, number, title in si_list:
        filename = f"{si_type}_{year}_{number}.xml"
        filepath = os.path.join(output_dir, filename)
        
        if os.path.exists(filepath):
            print(f"  [SKIP] {filename} (already exists)")
            success += 1
            continue
        
        url = f"https://www.legislation.gov.uk/{si_type}/{year}/{number}/data.xml"
        
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            if r.status_code == 200 and r.content.strip().startswith(b"<"):
                with open(filepath, "wb") as f:
                    f.write(r.content)
                size_kb = len(r.content) / 1024
                print(f"  [OK] {title} ({size_kb:.0f} KB)")
                success += 1
            else:
                print(f"  [ERR] HTTP {r.status_code} for {title}")
        except Exception as e:
            print(f"  [ERR] {title}: {e}")
        
        time.sleep(POLITE_DELAY)
    
    print(f"\n  ✅ Downloaded {success}/{len(si_list)} SIs")
    return success


# ============================================================
# 3. DOWNLOAD CASE LAW (National Archives)
# ============================================================

def download_case_law_search(keyword, courts, years, max_per_court=10, output_dir=None):
    """Search National Archives for transport-related cases.
    
    Uses the Find Case Law API to search by keyword, then downloads XML.
    Fallback: sequential number download for each court/year.
    """
    print(f"\n{'='*60}")
    print(f"⚖️  DOWNLOADING CASE LAW (keyword: '{keyword}')")
    print(f"{'='*60}")
    
    success = 0
    
    for court in courts:
        for year in years:
            fails = 0
            print(f"\n  --- {court.upper()} {year} ---")
            
            for num in range(1, max_per_court + 1):
                filename = f"{court.replace('/', '_')}_{year}_{num}.xml"
                filepath = os.path.join(output_dir, filename)
                
                if os.path.exists(filepath):
                    print(f"    [SKIP] {filename}")
                    success += 1
                    continue
                
                # Try multiple URL patterns
                urls = [
                    f"https://caselaw.nationalarchives.gov.uk/{court}/{year}/{num}/data.xml",
                    f"https://caselaw.nationalarchives.gov.uk/{court}/{year}/{num}.xml",
                ]
                
                downloaded = False
                for url in urls:
                    try:
                        r = requests.get(url, headers=HEADERS, timeout=10)
                        if r.status_code == 200 and r.content.strip().startswith(b"<"):
                            with open(filepath, "wb") as f:
                                f.write(r.content)
                            print(f"    [OK] {filename}")
                            success += 1
                            downloaded = True
                            fails = 0
                            break
                    except Exception:
                        pass
                
                if not downloaded:
                    fails += 1
                    if fails >= 5:
                        print(f"    >> Skipping {court} {year} (5 consecutive misses)")
                        break
                
                time.sleep(POLITE_DELAY)
    
    print(f"\n  ✅ Downloaded {success} cases")
    return success


# ============================================================
# 4. DOWNLOAD AMENDMENTS TABLE (Structured Data)
# ============================================================

def download_amendments_table(acts_list, output_dir):
    """Download the 'changes to legislation' table for each Act.
    
    legislation.gov.uk provides structured data about which Acts
    amend, repeal, or modify each other. This is gold for the KG —
    it gives us ground-truth amendment relationships we can validate against.
    
    URL pattern: /ukpga/YEAR/NUM/data.xht?view=extent&timeline=true
    Also try: /ukpga/YEAR/NUM/changes/affected
    """
    print(f"\n{'='*60}")
    print(f"🔗 DOWNLOADING AMENDMENTS TABLES")
    print(f"{'='*60}")
    
    success = 0
    for act_type, year, number, title in acts_list:
        filename = f"{act_type}_{year}_{number}_amendments.json"
        filepath = os.path.join(output_dir, filename)
        
        if os.path.exists(filepath):
            print(f"  [SKIP] {filename}")
            success += 1
            continue
        
        # Try the changes feed (structured data about what amends this Act)
        url = f"https://www.legislation.gov.uk/{act_type}/{year}/{number}/changes/affected/data.feed"
        
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            if r.status_code == 200:
                with open(filepath.replace('.json', '.xml'), "wb") as f:
                    f.write(r.content)
                print(f"  [OK] {title} amendments feed")
                success += 1
            else:
                print(f"  [SKIP] No amendments feed for {title} (HTTP {r.status_code})")
        except Exception as e:
            print(f"  [ERR] {title}: {e}")
        
        time.sleep(POLITE_DELAY)
    
    print(f"\n  ✅ Downloaded {success} amendments tables")
    return success


# ============================================================
# 5. DOWNLOAD EXPLANATORY NOTES
# ============================================================

def download_explanatory_notes(acts_list, output_dir):
    """Download explanatory notes for Acts.
    
    Explanatory notes provide commentary on what each section does,
    which is extremely valuable for extracting detail_text.
    URL: /ukpga/YEAR/NUM/notes/data.xml
    """
    print(f"\n{'='*60}")
    print(f"📝 DOWNLOADING EXPLANATORY NOTES")
    print(f"{'='*60}")
    
    success = 0
    for act_type, year, number, title in acts_list:
        if act_type != "ukpga":
            continue  # Only primary Acts have explanatory notes
        
        filename = f"{act_type}_{year}_{number}_notes.xml"
        filepath = os.path.join(output_dir, filename)
        
        if os.path.exists(filepath):
            print(f"  [SKIP] {filename}")
            success += 1
            continue
        
        url = f"https://www.legislation.gov.uk/{act_type}/{year}/{number}/notes/data.xml"
        
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            if r.status_code == 200 and len(r.content) > 500:
                with open(filepath, "wb") as f:
                    f.write(r.content)
                size_kb = len(r.content) / 1024
                print(f"  [OK] {title} notes ({size_kb:.0f} KB)")
                success += 1
            else:
                print(f"  [SKIP] No notes for {title}")
        except Exception as e:
            print(f"  [ERR] {title}: {e}")
        
        time.sleep(POLITE_DELAY)
    
    print(f"\n  ✅ Downloaded {success} explanatory notes")
    return success


# ============================================================
# MAIN
# ============================================================

def main():
    print(f"""
╔══════════════════════════════════════════════════════════╗
║  LegalKGent — Transportation Law Data Download          ║
║  Domain: UK Transport, Road Traffic, Automated Vehicles ║
║  Sources: legislation.gov.uk + National Archives        ║
╚══════════════════════════════════════════════════════════╝
    """)
    
    start = time.time()
    stats = {}
    
    # 1. Primary legislation
    stats['acts'] = download_legislation(TRANSPORT_ACTS, LEGISLATION_DIR)
    
    # 2. Statutory Instruments
    stats['sis'] = download_statutory_instruments(TRANSPORT_SIS, SI_DIR)
    
    # 3. Case law (transport-related courts)
    stats['cases'] = download_case_law_search(
        keyword="transport",
        courts=["ewhc/admin", "ewca/civ", "uksc"],
        years=[2023, 2024],
        max_per_court=10,
        output_dir=CASELAW_DIR
    )
    
    # 4. Amendments tables (ground-truth for validation)
    stats['amendments'] = download_amendments_table(TRANSPORT_ACTS, AMENDMENTS_DIR)
    
    # 5. Explanatory notes
    stats['notes'] = download_explanatory_notes(TRANSPORT_ACTS, NOTES_DIR)
    
    elapsed = time.time() - start
    
    print(f"\n{'='*60}")
    print(f"🏁 DOWNLOAD COMPLETE ({elapsed:.0f}s)")
    print(f"{'='*60}")
    print(f"  Acts:              {stats['acts']}")
    print(f"  SIs:               {stats['sis']}")
    print(f"  Cases:             {stats['cases']}")
    print(f"  Amendments tables: {stats['amendments']}")
    print(f"  Explanatory notes: {stats['notes']}")
    print(f"\n  Files saved to: {OUTPUT_DIR}/")
    
    # Save download manifest
    manifest = {
        "download_time": datetime.now().isoformat(),
        "domain": "UK Transportation Law",
        "stats": stats,
        "acts": [{"type": t, "year": y, "number": n, "title": title}
                 for t, y, n, title in TRANSPORT_ACTS],
        "sis": [{"type": t, "year": y, "number": n, "title": title}
                for t, y, n, title in TRANSPORT_SIS],
    }
    with open(os.path.join(OUTPUT_DIR, "download_manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"  Manifest: {OUTPUT_DIR}/download_manifest.json")


if __name__ == "__main__":
    main()



╔══════════════════════════════════════════════════════════╗
║  LegalKGent — Transportation Law Data Download          ║
║  Domain: UK Transport, Road Traffic, Automated Vehicles ║
║  Sources: legislation.gov.uk + National Archives        ║
╚══════════════════════════════════════════════════════════╝
    

📜 DOWNLOADING PRIMARY LEGISLATION
  [OK] Automated Vehicles Act 2024 (1934 KB)
  [OK] Pedicabs (London) Act 2024 (8 KB)
  [OK] Road Traffic Act 1988 (4443 KB)
  [OK] Road Traffic Offenders Act 1988 (2962 KB)
  [OK] Road Traffic Regulation Act 1984 (3410 KB)
  [OK] Road Safety Act 2006 (1057 KB)
  [OK] Transport Act 2000 (6950 KB)
  [OK] Railways Act 1993 (4026 KB)
  [OK] Railways Act 2005 (1828 KB)
  [OK] Transport (London) Act 2008 (1499 KB)
  [OK] Air Traffic Management and Unmanned Aircraft Act 2021 (661 KB)
  [OK] Civil Aviation Act 2012 (1424 KB)
  [OK] Automated and Electric Vehicles Act 2018 (203 KB)
  [OK] Taxis and Private Hire Vehicles Act 2022 (69 KB)
  [OK] Highways Act 

In [1]:

# -*- coding: utf-8 -*-
"""
LegalKGent — Robust Knowledge Graph Creation Pipeline
======================================================
Smart XML chunking + Normalized triple extraction + Neo4j ingestion.
Copy each CELL block into a separate Jupyter/Colab cell.
"""

################################################################
# CELL 1 — Installs & Ollama Setup (Kaggle / Colab)
# Run this cell first. Works on both Kaggle and Colab.
################################################################

import os, subprocess, time



In [2]:
# -*- coding: utf-8 -*-
"""
LegalKGent — Robust Knowledge Graph Creation Pipeline
======================================================
Smart XML chunking + Normalized triple extraction + Neo4j ingestion.
Copy each CELL block into a separate Jupyter/Colab cell.
"""

################################################################
# CELL 1 — vLLM Setup (RunPod / Cloud GPU)
# Run this cell first. Then start vLLM server in a terminal.
################################################################

# --- Terminal Setup (run BEFORE starting the notebook) ---
# pip install vllm openai
#
# vllm serve Qwen/Qwen2.5-7B-Instruct \
#   --dtype auto \
#   --max-model-len 4096 \
#   --gpu-memory-utilization 0.90 \
#   --enable-prefix-caching \
#   --max-num-seqs 16 \
#   --port 8000
#
# Key flags:
#   --gpu-memory-utilization 0.90  → uses 90% of A40 VRAM (41GB/46GB)
#   --max-num-seqs 16             → 16 concurrent sequences in a batch
#   --enable-prefix-caching       → caches system prompt KV (huge speedup!)
#   --max-model-len 4096          → matches our chunk size
#
# For larger models:
#   Qwen/Qwen2.5-14B-Instruct  → fits on A40 with --gpu-memory-utilization 0.90
#   Qwen/Qwen2.5-32B-Instruct  → fits on A40 with --max-num-seqs 4
#
# Verify server is running:
#   curl http://localhost:8000/v1/models
#
# print('✅ vLLM server should be running on port 8000')

################################################################
# CELL 2 — Imports & Configuration
################################################################

import os
import json
import re
import time
import xml.etree.ElementTree as ET
from collections import defaultdict

# --- PATHS ---
RAW_LEGISLATION_DIR = "data/raw_legislation"
RAW_CASELAW_DIR = "data/raw_caselaw"
RAW_SI_DIR = "data/raw_statutory_instruments"
CORPUS_FILE = "data/legal_corpus_final.json"
SMART_CORPUS_FILE = "data/smart_corpus.json"
OUTPUT_FILE = "data/extracted_triples.json"
MODEL_NAME = "qwen2.5:7b"

# --- GPU OPTIMIZATION (vLLM on RunPod / A40) ---
# vLLM handles GPU optimization automatically:
#   - PagedAttention: efficient KV cache, no VRAM waste
#   - Continuous batching: processes requests as they arrive
#   - Prefix caching: system prompt computed once, reused for all chunks
#   - Flash Attention 2: enabled by default on Ampere GPUs (A40)
#
# VRAM budget (A40 48GB) with --gpu-memory-utilization 0.90:
#   Qwen2.5-7B:  ~41GB available → model ~5GB + KV cache ~36GB → 16+ concurrent seqs
#   Qwen2.5-14B: ~41GB available → model ~10GB + KV cache ~31GB → 8+ concurrent seqs
#   Qwen2.5-32B: ~41GB available → model ~20GB + KV cache ~21GB → 4+ concurrent seqs

# --- vLLM CONFIG ---
VLLM_BASE_URL = "http://localhost:8000/v1"
VLLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"  # Must match the model served by vLLM

BATCH_SIZE = 5000

# --- CANONICAL ACTIONS ---
# The ONLY valid relationship types in our KG
CANONICAL_ACTIONS = [
    # Legislative modification
    "AMENDS", "REPEALS", "SUBSTITUTES", "INSERTS",
    "COMMENCES", "REVOKES", "APPLIES", "CITES", "OVERRULES",
    # Semantic (SIs, Explanatory Notes, cross-domain)
    "DEFINES", "INTERPRETS", "DELEGATES", "IMPLEMENTS",
    # Power & obligation (works across all legal domains)
    "CREATES", "EMPOWERS", "REQUIRES", "PROHIBITS", "EXTENDS",
]

# Map any LLM variation to the canonical form
ACTION_NORMALIZER = {
    "AMEND": "AMENDS", "AMENDED": "AMENDS", "AMENDS": "AMENDS", "AMENDING": "AMENDS",
    "REPEAL": "REPEALS", "REPEALED": "REPEALS", "REPEALS": "REPEALS", "REPEALING": "REPEALS",
    "OMIT": "REPEALS", "OMITS": "REPEALS", "OMITTED": "REPEALS",  # "is omitted" = repeal
    "SUBSTITUTE": "SUBSTITUTES", "SUBSTITUTED": "SUBSTITUTES", "SUBSTITUTES": "SUBSTITUTES",
    "REPLACE": "SUBSTITUTES", "REPLACES": "SUBSTITUTES", "REPLACED": "SUBSTITUTES",
    "INSERT": "INSERTS", "INSERTED": "INSERTS", "INSERTS": "INSERTS", "INSERTING": "INSERTS",
    "COMMENCE": "COMMENCES", "COMMENCED": "COMMENCES", "COMMENCES": "COMMENCES",
    "REVOKE": "REVOKES", "REVOKED": "REVOKES", "REVOKES": "REVOKES",
    "APPLY": "APPLIES", "APPLIED": "APPLIES", "APPLIES": "APPLIES",
    "CITE": "CITES", "CITED": "CITES", "CITES": "CITES", "CITING": "CITES",
    "OVERRULE": "OVERRULES", "OVERRULED": "OVERRULES", "OVERRULES": "OVERRULES",
    "RELATES_TO": "CITES",  # fallback
    # Extended actions
    "DEFINE": "DEFINES", "DEFINED": "DEFINES", "DEFINES": "DEFINES", "DEFINING": "DEFINES",
    "INTERPRET": "INTERPRETS", "INTERPRETED": "INTERPRETS", "INTERPRETS": "INTERPRETS",
    "DELEGATE": "DELEGATES", "DELEGATED": "DELEGATES", "DELEGATES": "DELEGATES",
    "IMPLEMENT": "IMPLEMENTS", "IMPLEMENTED": "IMPLEMENTS", "IMPLEMENTS": "IMPLEMENTS",
    "TRANSPOSES": "IMPLEMENTS", "TRANSPOSED": "IMPLEMENTS",  # EU law transposition
    # Power & obligation actions
    "CREATE": "CREATES", "CREATED": "CREATES", "CREATES": "CREATES", "CREATING": "CREATES",
    "ESTABLISH": "CREATES", "ESTABLISHES": "CREATES", "ESTABLISHED": "CREATES",
    "EMPOWER": "EMPOWERS", "EMPOWERED": "EMPOWERS", "EMPOWERS": "EMPOWERS",
    "AUTHORISE": "EMPOWERS", "AUTHORISES": "EMPOWERS", "AUTHORIZE": "EMPOWERS",
    "CONFER": "EMPOWERS", "CONFERS": "EMPOWERS", "CONFERRED": "EMPOWERS",  # confers power
    "REQUIRE": "REQUIRES", "REQUIRED": "REQUIRES", "REQUIRES": "REQUIRES",
    "MANDATE": "REQUIRES", "MANDATES": "REQUIRES", "OBLIGATE": "REQUIRES",
    "IMPOSE": "REQUIRES", "IMPOSES": "REQUIRES",  # imposes duty
    "PROHIBIT": "PROHIBITS", "PROHIBITED": "PROHIBITS", "PROHIBITS": "PROHIBITS",
    "RESTRICT": "PROHIBITS", "RESTRICTS": "PROHIBITS", "FORBID": "PROHIBITS",
    "BAN": "PROHIBITS", "BANS": "PROHIBITS",
    "EXTEND": "EXTENDS", "EXTENDED": "EXTENDS", "EXTENDS": "EXTENDS", "EXTENDING": "EXTENDS",
    "RENEW": "EXTENDS", "RENEWS": "EXTENDS", "RENEWED": "EXTENDS",  # renews sunset clause
    "PROLONG": "EXTENDS", "PROLONGS": "EXTENDS",
}

print("✅ Cell 2 done — Config loaded")


✅ Cell 2 done — Config loaded


In [3]:
################################################################
# CELL 3 — Smart XML Flattener (Part A)
# Re-parses raw XML to extract CLML structural metadata
################################################################

# CLML namespaces — needed because ElementTree requires full namespace URIs
LEG_NS = 'http://www.legislation.gov.uk/namespaces/legislation'
META_NS = 'http://www.legislation.gov.uk/namespaces/metadata'
DC_NS = 'http://purl.org/dc/elements/1.1/'
AKN_NS = 'http://docs.oasis-open.org/legaldocml/ns/akn/3.0'
TNA_NS = 'https://caselaw.nationalarchives.gov.uk/akn'

NAMESPACES = {
    'leg': LEG_NS,
    'ukm': META_NS,
    'dc': DC_NS,
    'akn': AKN_NS,
    'tna': TNA_NS,
}


def _leg(tag):
    """Prefix a tag with the legislation namespace."""
    return f'{{{LEG_NS}}}{tag}'


def _akn(tag):
    """Prefix a tag with the AKN namespace."""
    return f'{{{AKN_NS}}}{tag}'


def extract_text_recursive(elem):
    """Extract all text from an XML element and its children, stripping tags."""
    parts = []
    if elem.text:
        parts.append(elem.text.strip())
    for child in elem:
        parts.append(extract_text_recursive(child))
        if child.tail:
            parts.append(child.tail.strip())
    return " ".join(p for p in parts if p)


def extract_defined_terms(root):
    """Find all <Term> definitions in the XML: 'the Act' -> 'Local Government Finance Act 1988'."""
    terms = {}
    for term_elem in root.iter(_leg('Term')):
        term_text = extract_text_recursive(term_elem)
        if term_text:
            # Try to find the full name nearby (parent text before the Term element)
            parent = None
            for p in root.iter():
                if term_elem in list(p):
                    parent = p
                    break
            if parent is not None:
                parent_text = extract_text_recursive(parent)
                # Pattern: "the Local Government Finance Act 1988 ("the Act")"
                match = re.search(
                    r'(?:the\s+)?([A-Z][A-Za-z\s,()]+?Act\s+\d{4})\s*\(\s*["\u201c]' + re.escape(term_text),
                    parent_text
                )
                if match:
                    terms[term_text] = match.group(1).strip()
    return terms


def extract_internal_links(section_elem):
    """Extract all <InternalLink> references from a section."""
    refs = []
    for link in section_elem.iter(_leg('InternalLink')):
        ref_text = extract_text_recursive(link)
        if ref_text:
            refs.append(ref_text)
    return refs


def extract_inline_amendments(section_elem):
    """Extract all <InlineAmendment> text from a section."""
    amendments = []
    for amend in section_elem.iter(_leg('InlineAmendment')):
        amend_text = extract_text_recursive(amend)
        if amend_text:
            amendments.append(amend_text)
    return amendments


def get_parent_pblock_title(elem, root):
    """Walk up the tree to find the nearest <Pblock><Title> ancestor."""
    parent_map = {c: p for p in root.iter() for c in p}

    current = elem
    while current is not None:
        tag_local = current.tag.split('}')[-1] if '}' in current.tag else current.tag
        if tag_local in ('Pblock', 'Part'):
            title_elem = current.find(_leg('Title'))
            if title_elem is not None:
                return extract_text_recursive(title_elem)
        current = parent_map.get(current)
    return None


def parse_legislation_xml(filepath):
    """Parse a CLML legislation XML file into smart chunks."""
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
    except ET.ParseError as e:
        print(f"    ⚠️ XML parse error: {filepath}: {e}")
        return []

    # Extract filename-based ID prefix
    filename = os.path.basename(filepath).replace('.xml', '')

    # Extract document title from PrimaryPrelims/Title (namespaced)
    doc_title = filename
    prelim_title = root.find(f'.//{_leg("PrimaryPrelims")}/{_leg("Title")}')
    if prelim_title is not None:
        doc_title = extract_text_recursive(prelim_title)
    else:
        # Fallback: any Title element
        title_elem = root.find(f'.//{_leg("Title")}')
        if title_elem is not None:
            doc_title = extract_text_recursive(title_elem)

    # Extract year from Number element or filename
    number_elem = root.find(f'.//{_leg("Number")}')
    year_text = extract_text_recursive(number_elem) if number_elem is not None else filename
    year_match = re.search(r'(\d{4})', year_text)
    year = year_match.group(1) if year_match else "unknown"

    # Extract document-level defined terms
    defined_terms = extract_defined_terms(root)

    # Extract date of enactment
    date_elem = root.find(f'.//{_leg("DateOfEnactment")}/{_leg("DateText")}')
    enactment_date = None
    if date_elem is not None:
        date_text = extract_text_recursive(date_elem)
        date_match = re.search(r'(\d{1,2})\w*\s+(\w+)\s+(\d{4})', date_text)
        if date_match:
            try:
                from datetime import datetime
                enactment_date = datetime.strptime(
                    f"{date_match.group(1)} {date_match.group(2)} {date_match.group(3)}",
                    "%d %B %Y"
                ).strftime("%Y-%m-%d")
            except ValueError:
                pass

    chunks = []

    # Find all provision elements (P1 = sections, P1group = grouped sections)
    for section_elem in root.iter():
        tag_local = section_elem.tag.split('}')[-1] if '}' in section_elem.tag else section_elem.tag
        if tag_local not in ('P1', 'P1group'):
            continue

        # Get section number
        pnum_elem = section_elem.find(f'.//{_leg("Pnumber")}')
        section_num = extract_text_recursive(pnum_elem) if pnum_elem is not None else None

        # Also check the element's attributes for section id
        doc_uri = section_elem.get('DocumentURI', '')
        id_attr = section_elem.get('id', '')

        # Build section identifier
        if section_num:
            section_id = section_num
        elif id_attr:
            section_id = id_attr.replace('section-', '').replace('schedule-', 'SCHEDULE ')
        else:
            continue

        chunk_id = f"{filename}.xml_{section_id}"

        # Get text content
        content = extract_text_recursive(section_elem)
        if not content or len(content) < 20:
            continue

        # Get section heading
        heading = None
        title_elem = section_elem.find(_leg('Title'))
        if title_elem is not None:
            heading = extract_text_recursive(title_elem)

        # Get structural metadata from XML attributes
        extent = section_elem.get('RestrictExtent', None)
        in_force_date = section_elem.get('RestrictStartDate', None)

        # Walk up to find parent block title
        part_title = get_parent_pblock_title(section_elem, root)

        # Extract inline amendments and cross-references
        internal_refs = extract_internal_links(section_elem)
        inline_amendments = extract_inline_amendments(section_elem)

        # Build the contextual content prefix
        content_prefix = f"ACT: {doc_title} ({year}) | SECTION: {section_id}"
        if part_title:
            content_prefix += f" | PART: {part_title}"
        if heading:
            content_prefix += f" | HEADING: {heading}"
        content_prefix += f" | TEXT: "

        chunk = {
            "id": chunk_id,
            "source": "legislation",
            "doc_title": doc_title,
            "year": year,
            "section": str(section_id),
            "part": part_title,
            "heading": heading,
            "extent": extent,
            "in_force_date": in_force_date or enactment_date,
            "internal_refs": internal_refs if internal_refs else None,
            "inline_amendments": inline_amendments if inline_amendments else None,
            "defined_terms": defined_terms if defined_terms else None,
            "content": content_prefix + content
        }
        chunks.append(chunk)

    # Also parse Schedules
    for schedule in root.iter(_leg('Schedule')):
        sched_num = schedule.find(_leg('Number'))
        sched_id = extract_text_recursive(sched_num) if sched_num is not None else "SCHEDULE"

        for para in schedule.iter(_leg('P1')):
            pnum = para.find(f'.//{_leg("Pnumber")}')
            para_num = extract_text_recursive(pnum) if pnum is not None else ""
            para_id = f"{sched_id}" + (f"_{para_num}" if para_num else "")
            chunk_id = f"{filename}.xml_{para_id}"

            content = extract_text_recursive(para)
            if not content or len(content) < 20:
                continue

            extent = para.get('RestrictExtent', schedule.get('RestrictExtent'))
            in_force = para.get('RestrictStartDate', schedule.get('RestrictStartDate'))

            chunk = {
                "id": chunk_id,
                "source": "legislation",
                "doc_title": doc_title,
                "year": year,
                "section": para_id,
                "part": sched_id,
                "heading": None,
                "extent": extent,
                "in_force_date": in_force or enactment_date,
                "internal_refs": extract_internal_links(para) or None,
                "inline_amendments": extract_inline_amendments(para) or None,
                "defined_terms": defined_terms if defined_terms else None,
                "content": f"ACT: {doc_title} ({year}) | SECTION: {para_id} | TEXT: {content}"
            }
            chunks.append(chunk)

    return chunks


def parse_caselaw_xml(filepath):
    """Parse a case law XML file into chunks (AKN namespace)."""
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
    except ET.ParseError as e:
        print(f"    ⚠️ Case law parse error: {filepath}: {e}")
        return []

    filename = os.path.basename(filepath).replace('.xml', '')

    # Extract case name — try multiple namespaced tags
    case_name = None

    # Try dc:title
    dc_title = root.find(f'.//{{{DC_NS}}}title')
    if dc_title is not None:
        case_name = extract_text_recursive(dc_title)

    # Try AKN FRBRname
    if not case_name:
        frbr_name = root.find(f'.//{_akn("FRBRname")}')
        if frbr_name is not None:
            case_name = frbr_name.get('value', None) or extract_text_recursive(frbr_name)

    if not case_name:
        case_name = filename

    # Extract year
    year_match = re.search(r'(\d{4})', filename)
    year = year_match.group(1) if year_match else "unknown"

    # Determine court level from filename
    court_level = "Unknown"
    if "uksc_" in filename:
        court_level = "Supreme Court"
    elif "ewca_" in filename:
        court_level = "Court of Appeal"
    elif "ewhc_" in filename:
        court_level = "High Court"

    # Extract paragraphs — try AKN namespace then bare
    chunks = []
    paragraphs = list(root.iter(_akn('paragraph'))) or list(root.iter(_akn('Paragraph')))

    # If no structured paragraphs, extract all text as one chunk
    if not paragraphs:
        full_text = extract_text_recursive(root)
        if full_text and len(full_text) > 50:
            chunks.append({
                "id": f"{filename}.xml_full",
                "source": "judgment",
                "doc_title": case_name,
                "year": year,
                "section": "full",
                "part": None,
                "heading": None,
                "extent": None,
                "in_force_date": None,
                "court_level": court_level,
                "internal_refs": None,
                "inline_amendments": None,
                "defined_terms": None,
                "content": f"CASE: {case_name} ({year}) | COURT: {court_level} | TEXT: {full_text}"
            })
        return chunks

    for i, para in enumerate(paragraphs):
        para_text = extract_text_recursive(para)
        if not para_text or len(para_text) < 30:
            continue

        para_num = para.get('Number', para.get('eId', str(i + 1)))

        chunks.append({
            "id": f"{filename}.xml_{para_num}",
            "source": "judgment",
            "doc_title": case_name,
            "year": year,
            "section": str(para_num),
            "part": None,
            "heading": None,
            "extent": None,
            "in_force_date": None,
            "court_level": court_level,
            "internal_refs": None,
            "inline_amendments": None,
            "defined_terms": None,
            "content": f"CASE: {case_name} ({year}) | COURT: {court_level} | PARA: {para_num} | TEXT: {para_text}"
        })

    return chunks


def build_smart_corpus():
    """Parse all raw XML files into smart chunks with CLML metadata."""
    all_chunks = []

    # Parse primary legislation
    if os.path.exists(RAW_LEGISLATION_DIR):
        xml_files = sorted([f for f in os.listdir(RAW_LEGISLATION_DIR) if f.endswith('.xml')])
        print(f"📂 Found {len(xml_files)} legislation XML files")
        for f in xml_files:
            chunks = parse_legislation_xml(os.path.join(RAW_LEGISLATION_DIR, f))
            all_chunks.extend(chunks)
            print(f"   {f}: {len(chunks)} chunks")
    else:
        print(f"⚠️ No legislation directory: {RAW_LEGISLATION_DIR}")

    # Parse statutory instruments (same CLML format as primary legislation)
    if os.path.exists(RAW_SI_DIR):
        xml_files = sorted([f for f in os.listdir(RAW_SI_DIR) if f.endswith('.xml')])
        print(f"📂 Found {len(xml_files)} statutory instrument XML files")
        for f in xml_files:
            chunks = parse_legislation_xml(os.path.join(RAW_SI_DIR, f))
            all_chunks.extend(chunks)
            print(f"   {f}: {len(chunks)} chunks")
    else:
        print(f"ℹ️  No SI directory: {RAW_SI_DIR} (run 1_download_data.py first)")

    # Parse case law
    if os.path.exists(RAW_CASELAW_DIR):
        xml_files = sorted([f for f in os.listdir(RAW_CASELAW_DIR) if f.endswith('.xml')])
        print(f"📂 Found {len(xml_files)} case law XML files")
        for f in xml_files:
            chunks = parse_caselaw_xml(os.path.join(RAW_CASELAW_DIR, f))
            all_chunks.extend(chunks)
            print(f"   {f}: {len(chunks)} chunks")
    else:
        print(f"⚠️ No case law directory: {RAW_CASELAW_DIR}")

    # Save
    with open(SMART_CORPUS_FILE, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Smart corpus built: {len(all_chunks)} chunks → {SMART_CORPUS_FILE}")
    return all_chunks


# Build or load smart corpus
if os.path.exists(SMART_CORPUS_FILE):
    print(f"📂 Loading existing smart corpus from {SMART_CORPUS_FILE}")
    with open(SMART_CORPUS_FILE, "r", encoding="utf-8") as f:
        smart_corpus = json.load(f)
    print(f"   {len(smart_corpus)} chunks loaded")
else:
    print("🔨 Building smart corpus from raw XML...")
    smart_corpus = build_smart_corpus()


📂 Loading existing smart corpus from data/smart_corpus.json
   15637 chunks loaded


In [4]:
################################################################
# CELL 4 — Corpus Stats & Quality Check
################################################################

sources = defaultdict(int)
with_date = 0
with_extent = 0
with_amendments = 0
with_terms = 0

for c in smart_corpus:
    sources[c.get('source', 'unknown')] += 1
    if c.get('in_force_date'):
        with_date += 1
    if c.get('extent'):
        with_extent += 1
    if c.get('inline_amendments'):
        with_amendments += 1
    if c.get('defined_terms'):
        with_terms += 1

print(f"📊 Smart Corpus Stats:")
print(f"   Total chunks: {len(smart_corpus)}")
print(f"   Sources: {dict(sources)}")
print(f"   With in_force_date: {with_date}")
print(f"   With extent: {with_extent}")
print(f"   With inline_amendments: {with_amendments}")
print(f"   With defined_terms: {with_terms}")

# Show a sample smart chunk
print(f"\n📝 Sample smart chunk:")
sample = next((c for c in smart_corpus if c.get('inline_amendments')), smart_corpus[0])
print(json.dumps({k: v for k, v in sample.items() if k != 'content'}, indent=2, default=str))
print(f"   content: {sample['content'][:200]}...")


################################################################
# CELL 5 — Normalization Functions (Part B)
################################################################

def normalize_action(raw_action):
    """Map any LLM action string to canonical form. Returns None if unrecognized."""
    if not raw_action:
        return None
    normalized = ACTION_NORMALIZER.get(raw_action.upper().strip())
    if normalized:
        return normalized
    # Fuzzy fallback: check if any canonical action is contained
    upper = raw_action.upper().strip()
    for canon in CANONICAL_ACTIONS:
        if canon in upper or upper in canon:
            return canon
    return None


def build_abbreviation_table(corpus):
    """
    Auto-extract abbreviation definitions from corpus text.
    Looks for patterns like: 'the Landlord and Tenant Act 1985 ("the LTA 1985")'
    Also uses <Term> definitions extracted during XML parsing.
    """
    abbrev_table = {}

    # 1. From defined_terms in smart chunks
    for chunk in corpus:
        terms = chunk.get('defined_terms')
        if terms and isinstance(terms, dict):
            for short, full in terms.items():
                if short and full and len(short) < len(full):
                    abbrev_table[short] = full

    # 2. From text patterns: '... Full Act Name Year ("abbreviation")'
    pattern = re.compile(
        r'((?:the\s+)?[A-Z][A-Za-z\s,\'-]+?Act\s+\d{4})\s*'
        r'\(\s*["\u201c]\s*((?:the\s+)?[A-Z][A-Za-z\s]+?\d{4})\s*["\u201d]\s*\)'
    )
    for chunk in corpus:
        content = chunk.get('content', '')
        for match in pattern.finditer(content):
            full_name = match.group(1).strip()
            abbreviation = match.group(2).strip()
            if abbreviation and full_name and len(abbreviation) < len(full_name):
                abbrev_table[abbreviation] = full_name

    print(f"📚 Abbreviation table: {len(abbrev_table)} entries")
    for short, full in sorted(abbrev_table.items()):
        print(f"   {short} → {full}")

    return abbrev_table


def build_id_to_title_map(corpus):
    """Map source ID prefixes to readable Act titles."""
    id_to_title = {}
    for c in corpus:
        chunk_id = c.get('id', '')
        prefix = chunk_id.rsplit('.xml_', 1)[0] if '.xml_' in chunk_id else chunk_id
        if prefix and prefix not in id_to_title:
            id_to_title[prefix] = c.get('doc_title', prefix)
    return id_to_title


def normalize_citation(raw_citation, abbrev_table=None):
    """
    Normalize a citation to consistent format.
    - Expand abbreviations (LRA 1967 → Leasehold Reform Act 1967)
    - Standardize section refs: 'section 5' → 's.5', 'Schedule 2' → 'Sch.2'
    """
    if not raw_citation:
        return None

    citation = raw_citation.strip()

    # 1. Expand known abbreviations
    if abbrev_table:
        for short, full in abbrev_table.items():
            if short in citation:
                citation = citation.replace(short, full)

    # 2. Normalize section references (but be careful not to break Act names)
    # Only normalize AFTER the Act name (year pattern)
    year_match = re.search(r'\d{4}', citation)
    if year_match:
        pos = year_match.end()
        act_part = citation[:pos]
        ref_part = citation[pos:]

        # Normalize reference part
        ref_part = re.sub(r'\bsection\s+', 's.', ref_part, flags=re.IGNORECASE)
        ref_part = re.sub(r'\bsections\s+', 'ss.', ref_part, flags=re.IGNORECASE)
        ref_part = re.sub(r'\bSchedule\s+', 'Sch.', ref_part, flags=re.IGNORECASE)
        ref_part = re.sub(r'\bSCHEDULE\s+', 'Sch.', ref_part)
        ref_part = re.sub(r'\bparagraph\s+', 'para.', ref_part, flags=re.IGNORECASE)
        ref_part = re.sub(r'\bparagraphs\s+', 'paras.', ref_part, flags=re.IGNORECASE)
        ref_part = re.sub(r'\bregulation\s+', 'reg.', ref_part, flags=re.IGNORECASE)
        ref_part = re.sub(r'\bregulations\s+', 'regs.', ref_part, flags=re.IGNORECASE)
        ref_part = re.sub(r'\barticle\s+', 'art.', ref_part, flags=re.IGNORECASE)

        citation = act_part + ref_part

    return citation.strip()


def extract_act_name(citation):
    """Pull out just the Act/Case name without section reference.
    'Housing Act 1996 s.122' → 'Housing Act 1996'
    """
    if not citation:
        return None
    # Match: words ending with Act/Bill/Order + year
    match = re.match(r'(.*?(?:Act|Bill|Order|Regulations?|Rules?)\s+\d{4})', citation)
    if match:
        return match.group(1).strip()
    return citation.strip()


# Build the lookup tables
abbrev_table = build_abbreviation_table(smart_corpus)
id_to_title = build_id_to_title_map(smart_corpus)

print(f"\n📋 ID → Title map: {len(id_to_title)} entries")
for sid, title in sorted(id_to_title.items())[:10]:
    print(f"   {sid} = {title}")
print("   ...")


📊 Smart Corpus Stats:
   Total chunks: 15637
   Sources: {'legislation': 11524, 'judgment': 4113}
   With in_force_date: 11496
   With extent: 4513
   With inline_amendments: 1737
   With defined_terms: 0

📝 Sample smart chunk:
{
  "id": "ukpga_1980_34.xml_62",
  "source": "legislation",
  "doc_title": "Transport Act 1980",
  "year": "1980",
  "section": "62",
  "part": "Miscellaneous and General",
  "heading": "Grants towards duty charged on bus fuel, and new bus grants.",
  "extent": "E+W+S",
  "in_force_date": "1980-06-30",
  "internal_refs": null,
  "inline_amendments": [
    "\u201c\n                  \u201c bus service \u201d\n                   means a stage carriage service within the meaning of Part I of the Transport Act 1980 which is available to the general public and is neither an excursion or tour within the meaning of that Part nor a service as regards which the condition specified in section 3(3)(a) of that Act (long journeys only) is satisfied;\n                \u201d"

In [5]:
################################################################
# CELL 6 — Improved System Prompt
################################################################

from openai import OpenAI

# Initialize vLLM client (OpenAI-compatible API)
vllm_client = OpenAI(base_url=VLLM_BASE_URL, api_key="not-needed")

SYSTEM_PROMPT = """You are a UK Legal Knowledge Engineer. Extract ALL legal relationships from the given text.

RELATIONSHIP TYPES (use EXACTLY these names):

== Legislative Modification ==
- AMENDS: modifies another law ("is amended", "for X substitute Y", "after X insert Y")
- REPEALS: removes/omits another law ("is repealed", "shall cease to have effect", "is omitted")
- SUBSTITUTES: replaces specific text/wording ("for 'X' substitute 'Y'")
- INSERTS: adds new provisions ("after section X insert")
- COMMENCES: brings a law into force ("comes into force on", "commencement order")
- REVOKES: removes secondary legislation ("is revoked")

== Judicial ==
- OVERRULES: court overrules a previous case ("overruled", "departed from")
- CITES: references another law or case without modifying it

== Semantic / Cross-Domain ==
- APPLIES: law applies to a scope ("applies to England and Wales", "applies to vehicles over 3.5t")
- DEFINES: creates a legal definition ("'automated vehicle' means...", "'road' has the meaning given by...")
- INTERPRETS: clarifies meaning ("section 5 is to be read as if...", "construed as")
- DELEGATES: grants regulation-making power to a body ("The Secretary of State may by regulations...")
- IMPLEMENTS: gives effect to/transposes ("implementing Directive 2019/...", "gives effect to")

== Power & Obligation (use these for what the law DOES, not just what it changes) ==
- CREATES: establishes a new body, offence, right, or role ("There is established a...", "it shall be an offence to...")
- EMPOWERS: grants powers ("may by regulations impose...", "has power to...", "authorises")
- REQUIRES: creates statutory duties/obligations ("the operator must...", "shall notify", "is required to")
- PROHIBITS: creates restrictions/offences ("no person shall...", "it is an offence to...", "must not")
- EXTENDS: extends temporal scope ("the levy period is extended to...", "remains in force until", "sunset date")

RULES:
1. Use formal FULL citations — e.g., "Welfare Reform Act 2012 s.3", "[2023] UKSC 1"
2. NEVER use abbreviations like "LRA 1967" or "the Act" — always expand to the full Act name
3. Capture effective_date in YYYY-MM-DD format if mentioned
4. For SUBSTITUTES, capture the NEW text in detail_text
5. For CREATES/EMPOWERS/REQUIRES/PROHIBITS, describe what is created/empowered/required/prohibited in detail_text
6. Return empty array [] if NO relationships found
7. Do NOT hallucinate relationships not explicitly stated in the text
8. Each relationship should have exactly ONE target — if text says "sections 5, 6 and 7 are repealed", create THREE separate relationships

BAD EXAMPLES (do NOT produce these):
- {"action": "INSERT", ...}  ← wrong, use "INSERTS"
- {"action": "REPLACES", ...}  ← wrong, use "SUBSTITUTES"
- {"action": "BANS", ...}  ← wrong, use "PROHIBITS"
- {"action": "ESTABLISHES", ...}  ← wrong, use "CREATES"
- {"target_citation": "the Act"}  ← wrong, use full name
- {"target_citation": "LRA 1967"}  ← wrong, use "Leasehold Reform Act 1967"
- {"target_citation": "sections 5, 6 and 7"}  ← wrong, split into separate objects

Respond with ONLY a JSON array. Each object must have:
{"action": "...", "target_citation": "...", "detail_text": "..." or null, "effective_date": "YYYY-MM-DD" or null}

Example output:
[{"action": "REPEALS", "target_citation": "Welfare Reform Act 2012 s.3", "detail_text": null, "effective_date": "2024-01-01"},
 {"action": "CREATES", "target_citation": "Road Traffic Act 1988 s.3A", "detail_text": "New offence of causing death by careless driving when under the influence", "effective_date": null},
 {"action": "EMPOWERS", "target_citation": "Automated Vehicles Act 2024 s.1", "detail_text": "Secretary of State may by regulations define criteria for self-driving vehicles", "effective_date": null}]"""

print("✅ Cell 6 done — System prompt defined")



✅ Cell 6 done — System prompt defined


In [6]:
################################################################
# CELL 6B — Graph-Aware Extraction Context (MedKGent Constructor Agent Pattern)
################################################################

def graph_context_for_chunk(chunk, neo4j_driver=None, max_context_triples=15):
    """Query Neo4j for existing triples related to Acts mentioned in this chunk.
    
    Inspired by MedKGent's Constructor Agent which checks the existing graph
    before inserting triples, giving the LLM context to:
    - Avoid contradicting existing knowledge
    - Produce richer detail_text
    - Maintain consistent entity naming
    
    Args:
        chunk: Dict with 'content', 'doc_title', etc.
        neo4j_driver: Neo4j driver instance (optional, returns empty if None)
        max_context_triples: Max number of existing triples to include
    
    Returns:
        String of existing triples formatted as context, or empty string.
    """
    if neo4j_driver is None:
        return ""
    
    # Extract Act names from chunk text
    content = chunk.get('content', '')
    doc_title = chunk.get('doc_title', '')
    
    # Find Act references in the text
    act_pattern = r'([A-Z][A-Za-z\s\(\)]+(?:Act|Order|Regulations?)\s+\d{4})'
    act_mentions = set(re.findall(act_pattern, content))
    if doc_title:
        act_mentions.add(doc_title)
    
    if not act_mentions:
        return ""
    
    context_lines = []
    try:
        with neo4j_driver.session() as session:
            for act_name in list(act_mentions)[:5]:  # Limit to 5 Acts
                # Search by source title or target citation
                result = session.run("""
                    MATCH (s:LegalDoc)-[r:LEGAL_RELATIONSHIP]->(t:LegalDoc)
                    WHERE s.title CONTAINS $act_name OR t.citation CONTAINS $act_name
                    RETURN s.id AS source, r.action_type AS action, 
                           t.citation AS target, r.detail AS detail
                    LIMIT $limit
                """, act_name=act_name.strip(), limit=max_context_triples)
                
                for record in result:
                    detail_part = f" | {record['detail'][:80]}" if record.get('detail') else ""
                    context_lines.append(
                        f"  [{record['action']}] {record['source']} -> {record['target']}{detail_part}"
                    )
    except Exception as e:
        print(f"    ⚠️ Graph context lookup failed: {e}")
        return ""
    
    if not context_lines:
        return ""
    
    # Deduplicate and limit
    context_lines = list(dict.fromkeys(context_lines))[:max_context_triples]
    
    return "EXISTING RELATIONSHIPS IN KNOWLEDGE GRAPH:\n" + "\n".join(context_lines)


def extract_triples_with_confidence(chunk, abbrev_table=None, neo4j_driver=None,
                                     n_samples=5, confidence_threshold=0.4, max_retries=2):
    """Extract triples using sampling-based confidence scoring (MedKGent style).
    
    Runs N parallel inferences per chunk with elevated temperature, counts
    frequency of each (action, target_citation) pair, and assigns
    confidence = frequency / N. Filters triples below threshold.
    
    Args:
        chunk: Source chunk dict
        abbrev_table: Abbreviation lookup table
        neo4j_driver: Optional Neo4j driver for graph-aware context
        n_samples: Number of inference runs (MedKGent uses 50, we use 5)
        confidence_threshold: Minimum confidence to keep (MedKGent uses 0.6)
        max_retries: Max retries per inference
    
    Returns:
        List of triples, each with added 'confidence' field.
    """
    from collections import Counter
    
    # Get graph context if Neo4j is available
    graph_context = graph_context_for_chunk(chunk, neo4j_driver)
    
    # Run N inferences with elevated temperature
    all_runs = []
    for i in range(n_samples):
        try:
            # Build prompt (same as extract_triples_vllm but with graph context)
            user_content = f"Extract all legal relationships from this text:\n\n"
            
            # Add graph context if available
            if graph_context:
                user_content += f"{graph_context}\n\n"
                user_content += "Use the above existing relationships to maintain consistency.\n\n"
            
            user_content += f"DOCUMENT ID: {chunk['id']}\n"
            user_content += f"TITLE: {chunk['doc_title']}\n"
            user_content += f"SOURCE TYPE: {chunk.get('source', 'unknown')}\n"
            
            if chunk.get('part'):
                user_content += f"PART: {chunk['part']}\n"
            if chunk.get('heading'):
                user_content += f"HEADING: {chunk['heading']}\n"
            if chunk.get('defined_terms'):
                user_content += f"DEFINED TERMS: {json.dumps(chunk['defined_terms'])}\n"
            if chunk.get('inline_amendments'):
                user_content += f"PRE-MARKED AMENDMENTS: {json.dumps(chunk['inline_amendments'][:5])}\n"
            
            user_content += f"\nTEXT:\n{chunk['content']}\n\n"
            user_content += "Respond with ONLY a JSON array of relationships. If none found, respond with []"
            
            response = vllm_client.chat.completions.create(
                model=VLLM_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_content}
                ],
                temperature=0.7,  # Higher temp for diversity
                max_tokens=2970,
                extra_body={"guided_json": None}  # Let model output freely
            )
            
            raw_text = response.choices[0].message.content.strip()
            
            # Parse JSON
            try:
                parsed = json.loads(raw_text)
            except json.JSONDecodeError:
                match = re.search(r'\[.*\]', raw_text, re.DOTALL)
                if match:
                    parsed = json.loads(match.group())
                else:
                    continue
            
            # Handle wrapped formats
            if isinstance(parsed, dict):
                items = (parsed.get("mutations") or parsed.get("relationships")
                         or parsed.get("results") or parsed.get("data") or [])
                if not items and "action" in parsed:
                    items = [parsed]
            elif isinstance(parsed, list):
                items = parsed
            else:
                continue
            
            # Normalize and collect
            run_triples = []
            for item in items:
                if not isinstance(item, dict):
                    continue
                if "action" not in item or "target_citation" not in item:
                    continue
                if not item["target_citation"]:
                    continue
                
                action = normalize_action(item["action"])
                if not action:
                    continue
                
                citation = normalize_citation(item["target_citation"], abbrev_table)
                if not citation or len(citation) < 3:
                    continue
                
                # Use (action, citation) as the key for frequency counting
                run_triples.append({
                    "action": action,
                    "target_citation": citation,
                    "detail_text": item.get("detail_text"),
                    "effective_date": item.get("effective_date"),
                })
            
            all_runs.append(run_triples)
            
        except Exception as e:
            print(f"    ⚠️ Confidence sample {i+1}/{n_samples} failed: {e}")
            continue
    
    if not all_runs:
        # Fallback to single-shot extraction
        print(f"    ⚠️ All confidence samples failed, falling back to single extraction")
        return extract_triples_vllm(chunk, abbrev_table, max_retries)
    
    # Count frequency of each (action, target_citation) pair
    triple_counts = Counter()
    triple_details = {}  # Store best detail_text and date for each key
    
    for run in all_runs:
        seen_in_run = set()  # Deduplicate within a single run
        for t in run:
            key = (t["action"], t["target_citation"])
            if key not in seen_in_run:
                triple_counts[key] += 1
                seen_in_run.add(key)
                # Keep detail_text from any run that provides it
                if key not in triple_details or (t.get("detail_text") and not triple_details[key].get("detail_text")):
                    triple_details[key] = t
    
    # Assign confidence = frequency / n_successful_runs and filter
    n_successful = len(all_runs)
    results = []
    
    source_prefix = chunk['id'].rsplit('.xml_', 1)[0] if '.xml_' in chunk['id'] else chunk['id']
    source_title = id_to_title.get(source_prefix, "")
    
    for key, count in triple_counts.items():
        confidence = count / n_successful
        
        if confidence < confidence_threshold:
            continue
        
        action, citation = key
        act_name = extract_act_name(citation)
        is_self = bool(source_title and act_name and source_title.lower() in act_name.lower())
        
        detail_info = triple_details.get(key, {})
        
        results.append({
            "action": action,
            "target_citation": citation,
            "target_act_name": act_name,
            "detail_text": detail_info.get("detail_text"),
            "effective_date": detail_info.get("effective_date"),
            "source_id": chunk['id'],
            "source_title": chunk.get('doc_title'),
            "source_section": chunk.get('section'),
            "in_force_date": chunk.get('in_force_date'),
            "extent": chunk.get('extent'),
            "is_self_amendment": is_self,
            "chunk_id": chunk['id'],
            "confidence": round(confidence, 2),
        })
    
    return results


In [16]:
################################################################
# CELL 7 — Extraction Function with Post-Processing
################################################################

def extract_triples_vllm(chunk, abbrev_table=None, max_retries=2):
    """Extract legal triples from a chunk using vLLM, with normalization."""

    # Build context-rich user prompt
    user_content = f"Extract all legal relationships from this text:\n\n"
    user_content += f"DOCUMENT ID: {chunk['id']}\n"
    user_content += f"TITLE: {chunk['doc_title']}\n"
    user_content += f"SOURCE TYPE: {chunk.get('source', 'unknown')}\n"

    if chunk.get('part'):
        user_content += f"PART: {chunk['part']}\n"
    if chunk.get('heading'):
        user_content += f"HEADING: {chunk['heading']}\n"
    if chunk.get('in_force_date'):
        user_content += f"IN FORCE DATE: {chunk['in_force_date']}\n"
    if chunk.get('extent'):
        user_content += f"EXTENT: {chunk['extent']}\n"
    if chunk.get('defined_terms'):
        user_content += f"DEFINED TERMS: {json.dumps(chunk['defined_terms'])}\n"
    if chunk.get('inline_amendments'):
        user_content += f"PRE-MARKED AMENDMENTS: {json.dumps(chunk['inline_amendments'][:5])}\n"

    user_content += f"\nTEXT:\n{chunk['content']}\n\n"
    user_content += "Respond with ONLY a JSON array of relationships. If none found, respond with []"

    for attempt in range(max_retries):
        try:
            response = vllm_client.chat.completions.create(
                model=VLLM_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_content}
                ],
                temperature=0.1,
                max_tokens=2970,
            )

            raw_text = response.choices[0].message.content.strip()

            # Parse JSON
            try:
                parsed = json.loads(raw_text)
            except json.JSONDecodeError:
                match = re.search(r'\[.*\]', raw_text, re.DOTALL)
                if match:
                    parsed = json.loads(match.group())
                else:
                    print(f"    ⚠️ Could not parse JSON: {raw_text[:100]}...")
                    return []

            # Handle wrapped formats
            if isinstance(parsed, dict):
                items = (parsed.get("mutations") or parsed.get("relationships")
                         or parsed.get("results") or parsed.get("data") or [])
                if not items and "action" in parsed:
                    items = [parsed]
            elif isinstance(parsed, list):
                items = parsed
            else:
                return []

            # Validate, normalize, and clean each triple
            results = []
            for item in items:
                if not isinstance(item, dict):
                    continue
                if "action" not in item or "target_citation" not in item:
                    continue
                if not item["target_citation"]:
                    continue

                # Normalize action
                action = normalize_action(item["action"])
                if not action:
                    print(f"    ⚠️ Unknown action '{item['action']}', skipping")
                    continue

                # Normalize citation
                citation = normalize_citation(item["target_citation"], abbrev_table)
                if not citation or len(citation) < 3:
                    continue

                # Extract parent act name
                act_name = extract_act_name(citation)

                # Detect self-amendment
                source_prefix = chunk['id'].rsplit('.xml_', 1)[0] if '.xml_' in chunk['id'] else chunk['id']
                source_title = id_to_title.get(source_prefix, "")
                is_self = bool(source_title and act_name and
                               source_title.lower() in act_name.lower())

                results.append({
                    "action": action,
                    "target_citation": citation,
                    "target_act_name": act_name,
                    "detail_text": item.get("detail_text"),
                    "effective_date": item.get("effective_date"),
                    "source_id": chunk['id'],
                    "source_title": chunk.get('doc_title'),
                    "source_section": chunk.get('section'),
                    "in_force_date": chunk.get('in_force_date'),
                    "extent": chunk.get('extent'),
                    "is_self_amendment": is_self,
                    "chunk_id": chunk['id'],  # link back to source chunk
                })

            return results

        except Exception as e:
            error_str = str(e).lower()
            if "timeout" in error_str or "connection" in error_str:
                print(f"    ⏳ Timeout (attempt {attempt+1}/{max_retries}), retrying...")
                time.sleep(2)
            else:
                print(f"    ❌ Error: {e}")
                return []

    print(f"    ❌ Failed after {max_retries} retries")
    return []


# Quick test
test_chunk = {
    "id": "test_act.xml_1",
    "doc_title": "Test Act 2024",
    "source": "legislation",
    "section": "1",
    "part": None,
    "heading": None,
    "extent": "E+W",
    "in_force_date": "2024-01-01",
    "defined_terms": None,
    "inline_amendments": None,
    "content": "Section 5 of the Welfare Reform Act 2012 is repealed."
}
test_result = extract_triples_vllm(test_chunk, abbrev_table)
print(f"🧪 Test extraction: {json.dumps(test_result, indent=2, default=str)}")
print("✅ Cell 7 done — Extraction with normalization working")


🧪 Test extraction: [
  {
    "action": "REPEALS",
    "target_citation": "Welfare Reform Act 2012 s.5",
    "target_act_name": "Welfare Reform Act 2012",
    "detail_text": null,
    "effective_date": "2024-01-01",
    "source_id": "test_act.xml_1",
    "source_title": "Test Act 2024",
    "source_section": "1",
    "in_force_date": "2024-01-01",
    "extent": "E+W",
    "is_self_amendment": false,
    "chunk_id": "test_act.xml_1"
  }
]
✅ Cell 7 done — Extraction with normalization working


In [17]:
################################################################
# CELL 8 — Build Batch
################################################################

def build_smart_batch(corpus, batch_size=5000):
    """Pick diverse chunks: amendment-heavy + repeal-heavy + caselaw + general."""
    repeal_idxs = [i for i, c in enumerate(corpus)
                   if 'repeal' in c.get('content', '').lower() or 'omit' in c.get('content', '').lower()]
    amend_idxs = [i for i, c in enumerate(corpus)
                  if 'amend' in c.get('content', '').lower() or 'substitut' in c.get('content', '').lower()]
    case_idxs = [i for i, c in enumerate(corpus) if c.get('source') == 'judgment']
    leg_idxs = [i for i, c in enumerate(corpus) if c.get('source') == 'legislation']

    selected = set()
    for idx in repeal_idxs:
        selected.add(idx)
    for idx in amend_idxs:
        selected.add(idx)
    for idx in case_idxs:
        selected.add(idx)
    for idx in leg_idxs:
        if len(selected) >= batch_size:
            break
        selected.add(idx)

    batch_indices = sorted(list(selected))[:batch_size]

    batch_sources = defaultdict(int)
    for idx in batch_indices:
        s = corpus[idx].get('source', 'unknown')
        batch_sources[s] += 1
    print(f"📋 Batch: {len(batch_indices)} chunks | {dict(batch_sources)}")
    return batch_indices

BATCH_SIZE=15500
batch_indices = build_smart_batch(smart_corpus, BATCH_SIZE)



📋 Batch: 15500 chunks | {'legislation': 11387, 'judgment': 4113}


In [ ]:
################################################################
# CELL 9 — Run Extraction (PARALLEL — uses all GPUs)
################################################################

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Load existing results
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        all_results = json.load(f)
    already_done = set(r['source_id'] for r in all_results if 'source_id' in r)
    print(f"📂 Loaded {len(all_results)} existing triples")
else:
    all_results = []
    already_done = set()

# Filter to chunks not yet processed
chunks_to_process = []
for idx in batch_indices:
    chunk = smart_corpus[idx]
    if chunk['id'] not in already_done:
        chunks_to_process.append(chunk)

NUM_WORKERS = 16  # vLLM handles batching — max-num-seqs=16 in vllm serve
SAVE_EVERY = 20   # save progress every N chunks

lock = threading.Lock()
stats = {"processed": 0, "triples_found": 0}
start_time = time.time()

print(f"\n🚀 Processing {len(chunks_to_process)} chunks with {MODEL_NAME} ({NUM_WORKERS} parallel workers)\n")


def process_chunk(chunk):
    """Extract triples from one chunk (thread-safe)."""
    triples = extract_triples_vllm(chunk, abbrev_table)
    return chunk, triples


with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {executor.submit(process_chunk, c): c for c in chunks_to_process}

    for future in as_completed(futures):
        chunk, triples = future.result()

        with lock:
            stats["processed"] += 1
            if triples:
                for t in triples:
                    all_results.append(t)
                    stats["triples_found"] += 1

                self_count = sum(1 for t in triples if t.get('is_self_amendment'))
                print(f"[{stats['processed']}/{len(chunks_to_process)}] {chunk['id']}: "
                      f"{len(triples)} triples ({self_count} self-amend)")
            else:
                print(f"[{stats['processed']}/{len(chunks_to_process)}] {chunk['id']}: (none)")

            # Save periodically
            if stats['processed'] % SAVE_EVERY == 0:
                with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                    json.dump(all_results, f, indent=2, ensure_ascii=False)
                elapsed_so_far = time.time() - start_time
                speed = stats['processed'] / elapsed_so_far
                remaining = (len(chunks_to_process) - stats['processed']) / max(speed, 0.01)
                print(f"   💾 Saved ({len(all_results)} triples) | "
                      f"{speed:.1f} chunks/sec | ~{remaining:.0f}s remaining")

# Final save
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

elapsed = time.time() - start_time
print(f"\n{'='*50}")
print(f"✅ DONE in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"   Processed: {stats['processed']}")
print(f"   Triples found: {stats['triples_found']}")
print(f"   Total in file: {len(all_results)}")
print(f"   Speed: {stats['processed']/max(elapsed,1):.1f} chunks/sec")


In [22]:
################################################################
# CELL 9 — Run Extraction (PARALLEL — uses all GPUs)
################################################################

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Load existing results
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        all_results = json.load(f)
    already_done = set(r['source_id'] for r in all_results if 'source_id' in r)
    print(f"📂 Loaded {len(all_results)} existing triples")
else:
    all_results = []
    already_done = set()

# Filter to chunks not yet processed
chunks_to_process = []
for idx in batch_indices:
    chunk = smart_corpus[idx]
    if chunk['id'] not in already_done:
        chunks_to_process.append(chunk)

NUM_WORKERS = 16  # vLLM handles batching — max-num-seqs=16 in vllm serve
SAVE_EVERY = 20   # save progress every N chunks

lock = threading.Lock()
stats = {"processed": 0, "triples_found": 0}
start_time = time.time()

print(f"\n🚀 Processing {len(chunks_to_process)} chunks with {MODEL_NAME} ({NUM_WORKERS} parallel workers)\n")


def process_chunk(chunk):
    """Extract triples from one chunk (thread-safe)."""
    triples = extract_triples_vllm(chunk, abbrev_table)
    return chunk, triples


with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {executor.submit(process_chunk, c): c for c in chunks_to_process}

    for future in as_completed(futures):
        chunk, triples = future.result()

        with lock:
            stats["processed"] += 1
            if triples:
                for t in triples:
                    all_results.append(t)
                    stats["triples_found"] += 1

                self_count = sum(1 for t in triples if t.get('is_self_amendment'))
                print(f"[{stats['processed']}/{len(chunks_to_process)}] {chunk['id']}: "
                      f"{len(triples)} triples ({self_count} self-amend)")
            else:
                print(f"[{stats['processed']}/{len(chunks_to_process)}] {chunk['id']}: (none)")

            # Save periodically
            if stats['processed'] % SAVE_EVERY == 0:
                with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                    json.dump(all_results, f, indent=2, ensure_ascii=False)
                elapsed_so_far = time.time() - start_time
                speed = stats['processed'] / elapsed_so_far
                remaining = (len(chunks_to_process) - stats['processed']) / max(speed, 0.01)
                print(f"   💾 Saved ({len(all_results)} triples) | "
                      f"{speed:.1f} chunks/sec | ~{remaining:.0f}s remaining")

# Final save
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

elapsed = time.time() - start_time
print(f"\n{'='*50}")
print(f"✅ DONE in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"   Processed: {stats['processed']}")
print(f"   Triples found: {stats['triples_found']}")
print(f"   Total in file: {len(all_results)}")
print(f"   Speed: {stats['processed']/max(elapsed,1):.1f} chunks/sec")

📂 Loaded 12574 existing triples

🚀 Processing 10576 chunks with qwen2.5:7b (16 parallel workers)

[1/10576] ukpga_1980_34.xml_33: (none)
[2/10576] ukpga_1980_34.xml_38: (none)
[3/10576] ukpga_1980_34.xml_43: (none)
[4/10576] ukpga_1980_34.xml_37: (none)
[5/10576] ukpga_1980_34.xml_44: (none)
[6/10576] ukpga_1980_34.xml_37: (none)
[7/10576] ukpga_1980_34.xml_42: (none)
[8/10576] ukpga_1980_34.xml_43: (none)
[9/10576] ukpga_1980_34.xml_42: (none)
[10/10576] ukpga_1980_34.xml_35: (none)
[11/10576] ukpga_1980_34.xml_46: (none)
[12/10576] ukpga_1980_34.xml_45: (none)
[13/10576] ukpga_1980_34.xml_45: (none)
[14/10576] ukpga_1980_34.xml_47: (none)
[15/10576] ukpga_1980_34.xml_46: (none)
[16/10576] ukpga_1980_34.xml_47: (none)
[17/10576] ukpga_1980_34.xml_48: (none)
[18/10576] ukpga_1980_34.xml_48: (none)
[19/10576] ukpga_1980_34.xml_52A: (none)
[20/10576] ukpga_1980_34.xml_57: (none)
   💾 Saved (12574 triples) | 7.0 chunks/sec | ~1505s remaining
[21/10576] ukpga_1980_34.xml_52B: (none)
[22/10


KeyboardInterrupt



    ⚠️ Could not parse JSON: [
    {"action": "DEFINES", "target_citation": "Transport Act 1980 s.53", "detail_text": null, "effe...
    ⚠️ Could not parse JSON: [
    {"action": "DEFINES", "target_citation": "Transport Act 1980 s.52B", "detail_text": null, "eff...
    ⚠️ Could not parse JSON: [
    {"action": "DEFINES", "target_citation": "Transport Act 1980 s.60", "detail_text": null, "effe...
    ⚠️ Could not parse JSON: [
    {"action": "DEFINES", "target_citation": "Transport Act 1980 s.60", "detail_text": null, "effe...
    ⚠️ Unknown action 'EXCLUDES', skipping
    ❌ Error: Expecting value: line 8 column 103 (char 2845)
    ⚠️ Unknown action 'EXCLUDES', skipping
    ⚠️ Unknown action 'EXCLUDES', skipping
    ⚠️ Unknown action 'EXCLUDES', skipping
    ⚠️ Could not parse JSON: [
    {"action": "DEFINES", "target_citation": "Road Traffic Regulation Act 1984 s.102", "detail_tex...
    ⚠️ Could not parse JSON: [
    {"action": "DEFINES", "target_citation": "Road Traffic Regulation Ac

In [19]:
print(f"   Processed: {stats['processed']}")

   Processed: 11375


In [24]:
stats['processed']

110

In [ ]:
################################################################
# CELL 9 — MANUAL RESUME (Starting at index 11375)
################################################################

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import json
import os
import time

# --- MANUAL SETTINGS ---
MANUAL_START_INDEX = 11375  # We skip the first 11,375 chunks
NUM_WORKERS = 16            # Matches your vLLM --max-num-seqs 16
SAVE_EVERY = 50             # Save every 50 chunks
# -----------------------

# 1. Load existing results to append to them
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        all_results = json.load(f)
    print(f"📂 Loaded {len(all_results)} existing triples from file.")
else:
    all_results = []
    print("📂 No existing file found. Starting fresh list.")

# 2. Slice the indices manually
# We take batch_indices and slice it to start exactly where you left off
remaining_indices = batch_indices[MANUAL_START_INDEX:]
chunks_to_process = [smart_corpus[idx] for idx in remaining_indices]

lock = threading.Lock()
stats = {
    "manual_offset": MANUAL_START_INDEX,
    "processed_this_session": 0, 
    "triples_found_this_session": 0
}
start_time = time.time()

print(f"📊 Manual Resume Status:")
print(f"   - Skipping the first: {MANUAL_START_INDEX} chunks")
print(f"   - Remaining to process: {len(chunks_to_process)} chunks")
print(f"🚀 Launching {NUM_WORKERS} workers...\n")

def process_chunk(chunk):
    """Thread-safe call to extraction function."""
    try:
        triples = extract_triples_vllm(chunk, abbrev_table)
        return chunk, triples
    except Exception as e:
        print(f"❌ Error on chunk {chunk.get('id')}: {e}")
        return chunk, []

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {executor.submit(process_chunk, c): c for c in chunks_to_process}

    for future in as_completed(futures):
        chunk, triples = future.result()

        with lock:
            stats["processed_this_session"] += 1
            # Calculate global progress for the print statement
            current_global_idx = stats["manual_offset"] + stats["processed_this_session"]
            
            if triples:
                for t in triples:
                    t['source_id'] = chunk['id'] # Tag for future-proofing
                    all_results.append(t)
                    stats["triples_found_this_session"] += 1

            # Progress tracking
            print(f"[{current_global_idx}/{len(batch_indices)}] {chunk['id']}: {len(triples)} triples")

            # Periodic Disk Save
            if stats['processed_this_session'] % SAVE_EVERY == 0:
                with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                    json.dump(all_results, f, indent=2, ensure_ascii=False)
                
                elapsed = time.time() - start_time
                speed = stats['processed_this_session'] / max(elapsed, 0.001)
                remaining_items = len(chunks_to_process) - stats['processed_this_session']
                rem_min = (remaining_items / speed) / 60 if speed > 0 else 0
                print(f"   💾 Saved! | Speed: {speed:.1f} chunks/sec | Est. Remaining: {rem_min:.1f} min")

# Final Save
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print(f"\n✅ Finished. Total triples in file: {len(all_results)}")

📂 Loaded 12630 existing triples from file.
📊 Manual Resume Status:
   - Skipping the first: 11375 chunks
   - Remaining to process: 4125 chunks
🚀 Launching 16 workers...

[11376/15500] uksi_2023_980.xml_1: 0 triples
[11377/15500] uksi_2023_980.xml_1: 0 triples
[11378/15500] uksi_2024_305.xml_1: 0 triples
[11379/15500] uksi_2024_305.xml_1: 0 triples
[11380/15500] uksi_2023_695.xml_2: 2 triples
[11381/15500] uksi_2023_980.xml_2: 1 triples
[11382/15500] ewca_civ_2023_1.xml_para_2: 0 triples
[11383/15500] ewca_civ_2023_1.xml_para_1: 0 triples
[11384/15500] ewca_civ_2023_1.xml_para_3: 0 triples
[11385/15500] ewca_civ_2023_1.xml_para_4: 0 triples
[11386/15500] uksi_2024_305.xml_2: 1 triples
[11387/15500] uksi_2023_695.xml_2: 3 triples
[11388/15500] uksi_2024_305.xml_2: 1 triples
[11389/15500] uksi_2023_980.xml_2: 2 triples
[11390/15500] ewca_civ_2023_1.xml_para_5: 0 triples
[11391/15500] ewca_civ_2023_1.xml_para_6: 0 triples
[11392/15500] ewca_civ_2023_1.xml_para_7: 0 triples
[11393/15500] e

In [27]:
print(f"   Processed: {stats['processed']}")


KeyError: 'processed'

In [28]:
stats

{'manual_offset': 11375,
 'processed_this_session': 4125,
 'triples_found_this_session': 308}

In [29]:
################################################################
# CELL 10 — Post-Processing: Dedup & Quality Report
################################################################

with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    all_triples = json.load(f)

print(f"📊 Post-Processing {len(all_triples)} triples...\n")

# 1. Deduplicate (same source_id + action + target_citation)
seen = set()
deduped = []
dup_count = 0
for t in all_triples:
    key = (t.get('source_id', ''), t.get('action', ''), t.get('target_citation', ''))
    if key not in seen:
        seen.add(key)
        deduped.append(t)
    else:
        dup_count += 1

print(f"   Duplicates removed: {dup_count}")

# 2. Re-normalize any remaining bad actions
fixed_actions = 0
for t in deduped:
    canonical = normalize_action(t.get('action', ''))
    if canonical and canonical != t.get('action'):
        t['action'] = canonical
        fixed_actions += 1
print(f"   Actions re-normalized: {fixed_actions}")

# 3. Re-normalize citations with abbreviation table
fixed_citations = 0
for t in deduped:
    old = t.get('target_citation', '')
    new = normalize_citation(old, abbrev_table)
    if new and new != old:
        t['target_citation'] = new
        t['target_act_name'] = extract_act_name(new)
        fixed_citations += 1
print(f"   Citations re-normalized: {fixed_citations}")

# Save cleaned version
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(deduped, f, indent=2, ensure_ascii=False)

print(f"\n✅ Cleaned: {len(deduped)} triples saved (was {len(all_triples)})")

# Quality report
print(f"\n{'='*50}")
print(f"📊 QUALITY REPORT")
print(f"{'='*50}")

actions = defaultdict(int)
for t in deduped:
    actions[t.get('action', 'UNKNOWN')] += 1
print(f"\n  Action distribution:")
for a, c in sorted(actions.items(), key=lambda x: -x[1]):
    marker = "✅" if a in CANONICAL_ACTIONS else "❌"
    print(f"    {marker} {a}: {c}")

self_amendments = sum(1 for t in deduped if t.get('is_self_amendment'))
print(f"\n  Self-amendments: {self_amendments}/{len(deduped)}")

with_date = sum(1 for t in deduped if t.get('effective_date'))
with_detail = sum(1 for t in deduped if t.get('detail_text'))
with_inforce = sum(1 for t in deduped if t.get('in_force_date'))
print(f"  With effective_date: {with_date}/{len(deduped)}")
print(f"  With detail_text: {with_detail}/{len(deduped)}")
print(f"  With in_force_date: {with_inforce}/{len(deduped)}")

vague = [t for t in deduped if t.get('target_citation') and len(t['target_citation']) < 15]
if vague:
    print(f"\n  ⚠️ Vague citations (<15 chars): {len(vague)}")
    for v in vague[:5]:
        print(f"      {v['source_id']} -> \"{v['target_citation']}\"")



📊 Post-Processing 12938 triples...

   Duplicates removed: 4207
   Actions re-normalized: 0
   Citations re-normalized: 0

✅ Cleaned: 8731 triples saved (was 12938)

📊 QUALITY REPORT

  Action distribution:
    ✅ SUBSTITUTES: 1467
    ✅ INSERTS: 1450
    ✅ AMENDS: 1138
    ✅ REQUIRES: 1070
    ✅ DEFINES: 991
    ✅ REPEALS: 883
    ✅ EMPOWERS: 556
    ✅ APPLIES: 456
    ✅ CREATES: 185
    ✅ INTERPRETS: 143
    ✅ PROHIBITS: 114
    ✅ DELEGATES: 108
    ✅ COMMENCES: 96
    ✅ CITES: 26
    ✅ OVERRULES: 24
    ✅ EXTENDS: 14
    ✅ REVOKES: 6
    ✅ IMPLEMENTS: 4

  Self-amendments: 5229/8731
  With effective_date: 7601/8731
  With detail_text: 6333/8731
  With in_force_date: 8435/8731

  ⚠️ Vague citations (<15 chars): 81
      ukpga_1980_34.xml_64 -> "Taxi"
      ukpga_1980_34.xml_52C -> "Minister"
      ukpga_1980_34.xml_52C -> "this section"
      ukpga_2005_14.xml_20 -> "Taxes Act"
      ukpga_2005_14.xml_12 -> "2001 Act"


In [30]:
################################################################
# CELL 11 — Results Analysis
################################################################

print(f"\n📝 Samples (one per action type):")
shown = set()
for t in deduped:
    if t['action'] not in shown:
        shown.add(t['action'])
        print(f"  [{t['action']}] {t.get('source_title','?')} s.{t.get('source_section','?')} -> {t['target_citation']}")
        if t.get('detail_text'):
            print(f"           detail: {t['detail_text'][:100]}...")





📝 Samples (one per action type):
  [REPEALS] Transport Act 1980 s.36 -> Transport Act 1960 s.144(1)
  [REQUIRES] Transport Act 1980 s.52 -> Transport Act 1980 s.52
           detail: Minister shall make payments to persons administering B.R. or N.F.C. pension schemes based on certai...
  [DEFINES] Transport Act 1980 s.51 -> Transport Act 1980 s.51
  [APPLIES] Transport Act 1980 s.51 -> enactments mentioned in Schedule 7
  [EMPOWERS] Transport Act 1980 s.59 -> Transport Act 1980 s.59
           detail: Board or Corporation can amend pension schemes to enable payment of increases or bring schemes into ...
  [SUBSTITUTES] Transport Act 1980 s.62 -> Finance Act 1965 s.92
           detail: bus service means a stage carriage service within the meaning of Part I of the Transport Act 1980 wh...
  [INSERTS] Transport Act 1980 s.62 -> Transport Act 1968 s.32
           detail: wholly or mainly in the operation of bus services...
  [CREATES] Transport Act 1980 s.54 -> Transport Act 1980 s.54
  

In [35]:
################################################################
# CELL 13 — Neo4j Ingestion (Improved Schema)
# Typed relationships + Chunk nodes
################################################################

from neo4j import GraphDatabase

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "LegalPassword123"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print(f"✅ Connected to Neo4j at {NEO4J_URI}")

# Load data
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    triples = json.load(f)
with open(SMART_CORPUS_FILE, "r", encoding="utf-8") as f:
    corpus_chunks = json.load(f)
print(f"📂 Loaded {len(triples)} triples and {len(corpus_chunks)} chunks")

# Clear existing graph
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("🗑️  Cleared existing graph")

# --- Step 1: Create Chunk nodes ---
print("\n📦 Creating Chunk nodes...")
chunk_count = 0
with driver.session() as session:
    for chunk in corpus_chunks:
        query = """
        MERGE (c:Chunk {id: $id})
        SET c.doc_title = $doc_title,
            c.source = $source,
            c.year = $year,
            c.section = $section,
            c.part = $part,
            c.heading = $heading,
            c.extent = $extent,
            c.in_force_date = $in_force_date,
            c.content = $content
        """
        session.run(query,
            id=chunk['id'],
            doc_title=chunk.get('doc_title'),
            source=chunk.get('source'),
            year=chunk.get('year'),
            section=chunk.get('section'),
            part=chunk.get('part'),
            heading=chunk.get('heading'),
            extent=chunk.get('extent'),
            in_force_date=chunk.get('in_force_date'),
            content=chunk.get('content', '')[:5000]  # Neo4j property size limit
        )
        chunk_count += 1
        if chunk_count % 500 == 0:
            print(f"   ... created {chunk_count} chunk nodes")
print(f"   ✅ Created {chunk_count} chunk nodes")

# --- Step 2: Create LegalDoc nodes + link chunks ---
print("\n📄 Creating LegalDoc nodes...")
with driver.session() as session:
    # Create LegalDoc for each source document (by prefix)
    for prefix, title in id_to_title.items():
        doc_type = "CaseLaw" if any(p in prefix for p in ["uksc_", "ewca_", "ewhc_", "ukut_"]) else "Legislation"
        year_match = re.search(r'(\d{4})', prefix)
        year = year_match.group(1) if year_match else None

        session.run("""
            MERGE (d:LegalDoc {id: $id})
            SET d.title = $title, d.type = $doc_type, d.year = $year
        """, id=prefix, title=title, doc_type=doc_type, year=year)

    # Link chunks to their parent LegalDoc
    session.run("""
        MATCH (c:Chunk), (d:LegalDoc)
        WHERE c.id STARTS WITH d.id + '.xml_'
        MERGE (c)-[:FROM_DOC]->(d)
    """)
print(f"   ✅ Created {len(id_to_title)} LegalDoc nodes with chunk links")

# --- Step 3: Create LEGAL_RELATIONSHIP edges with action_type property ---
# NOTE: Uses a single :LEGAL_RELATIONSHIP edge type (not typed edges like :AMENDS)
#       so that kg_query.py can query with: MATCH ()-[r:LEGAL_RELATIONSHIP]->()
#       The action is stored as r.action_type (e.g., 'AMENDS', 'REPEALS')
print("\n🔗 Creating LEGAL_RELATIONSHIP edges...")
loaded = 0
skipped = 0

with driver.session() as session:
    for t in triples:
        source_prefix = t.get('source_id', '').rsplit('.xml_', 1)[0] if '.xml_' in t.get('source_id', '') else t.get('source_id', '')
        target_citation = t.get('target_citation')
        action = t.get('action', 'CITES')

        if not target_citation:
            skipped += 1
            continue

        # Ensure action is canonical
        if action not in CANONICAL_ACTIONS:
            action = normalize_action(action) or 'CITES'

        target_act = t.get('target_act_name', target_citation)
        confidence = t.get('confidence', 1.0)  # Default 1.0 for legacy triples without confidence

        # Create source/target LegalDoc nodes and LEGAL_RELATIONSHIP edge
        query = """
        MERGE (s:LegalDoc {id: $source_id})
        SET s.title = $source_title, s.type = $source_type
        MERGE (t:LegalDoc {citation: $target_citation})
        SET t.act_name = $target_act
        MERGE (s)-[r:LEGAL_RELATIONSHIP {chunk_id: $chunk_id, action_type: $action_type}]->(t)
        SET r.detail = $detail,
            r.date = $date,
            r.in_force_date = $in_force_date,
            r.extent = $extent,
            r.is_self_amendment = $is_self,
            r.confidence = $confidence
        """

        source_type = "CaseLaw" if any(p in source_prefix for p in ["uksc_", "ewca_", "ewhc_"]) else "Legislation"

        session.run(query,
            source_id=source_prefix,
            source_title=t.get('source_title', source_prefix),
            source_type=source_type,
            target_citation=target_citation,
            target_act=target_act,
            action_type=action,
            chunk_id=t.get('chunk_id', ''),
            detail=t.get('detail_text'),
            date=t.get('effective_date'),
            in_force_date=t.get('in_force_date'),
            extent=t.get('extent'),
            is_self=t.get('is_self_amendment', False),
            confidence=confidence,
        )
        loaded += 1
        if loaded % 100 == 0:
            print(f"   ... loaded {loaded} relationships")

print(f"\n{'='*50}")
print(f"✅ NEO4J INGESTION COMPLETE")
print(f"   Loaded: {loaded} relationships")
print(f"   Skipped (null targets): {skipped}")

# --- Verify ---
with driver.session() as session:
    nodes = session.run("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS cnt").data()
    edges = session.run("""
        MATCH ()-[r]->()
        RETURN type(r) AS rel_type, count(r) AS cnt
        ORDER BY cnt DESC
    """).data()

print(f"\n📊 Graph Stats:")
for n in nodes:
    print(f"   {n['label']} nodes: {n['cnt']}")
print(f"\n   Relationship types:")
for e in edges:
    print(f"   {e['rel_type']}: {e['cnt']}")

driver.close()
print("\n✅ Neo4j connection closed")


✅ Connected to Neo4j at bolt://localhost:7687
📂 Loaded 8731 triples and 15637 chunks
🗑️  Cleared existing graph

📦 Creating Chunk nodes...
   ... created 500 chunk nodes
   ... created 1000 chunk nodes
   ... created 1500 chunk nodes
   ... created 2000 chunk nodes
   ... created 2500 chunk nodes
   ... created 3000 chunk nodes
   ... created 3500 chunk nodes
   ... created 4000 chunk nodes
   ... created 4500 chunk nodes
   ... created 5000 chunk nodes
   ... created 5500 chunk nodes
   ... created 6000 chunk nodes
   ... created 6500 chunk nodes
   ... created 7000 chunk nodes
   ... created 7500 chunk nodes
   ... created 8000 chunk nodes
   ... created 8500 chunk nodes
   ... created 9000 chunk nodes
   ... created 9500 chunk nodes
   ... created 10000 chunk nodes
   ... created 10500 chunk nodes
   ... created 11000 chunk nodes
   ... created 11500 chunk nodes
   ... created 12000 chunk nodes
   ... created 12500 chunk nodes
   ... created 13000 chunk nodes
   ... created 13500 ch

ModuleNotFoundError: No module named 'pyvis'

In [37]:
"""
LegalKGent — Improved Neo4j Ingestion Script
=============================================
Ingests extracted triples into Neo4j with:
  - Single :LEGAL_RELATIONSHIP edge type with action_type property
  - Confidence accumulation: s = 1 - (1-s)(1-s') when same triple re-appears
  - Provenance tracking: source_ids list on edges
  - Compatible with kg_query.py ReAct agent

Usage: Copy cells into Jupyter notebook or run as standalone script.
"""

from neo4j import GraphDatabase
import json

# ⚠️ UPDATE THESE with your Neo4j credentials
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "LegalPassword123"

# --- Connect ---
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print(f"✅ Connected to Neo4j at {NEO4J_URI}")

# --- Load triples ---
TRIPLES_FILE = "data/extracted_triples.json"
with open(TRIPLES_FILE, "r") as f:
    triples = json.load(f)
print(f"📂 Loaded {len(triples)} triples from {TRIPLES_FILE}")

# --- Clear existing graph (fresh start) ---
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("🗑️  Cleared existing graph")

# --- Create indexes for performance ---
with driver.session() as session:
    session.run("CREATE INDEX IF NOT EXISTS FOR (n:LegalDoc) ON (n.id)")
    session.run("CREATE INDEX IF NOT EXISTS FOR (n:LegalDoc) ON (n.citation)")
print("📇 Created indexes on LegalDoc.id and LegalDoc.citation")


# ============================================================
# Confidence accumulation formula (from MedKGent paper)
# s_new = 1 - (1 - s_existing) * (1 - s_incoming)
# 
# This means: if a triple appears multiple times, its confidence
# increases monotonically. E.g.:
#   First time:  0.8
#   Second time:  1 - (1-0.8)*(1-0.6) = 1 - 0.2*0.4 = 0.92
#   Third time:   1 - (1-0.92)*(1-0.7) = 1 - 0.08*0.3 = 0.976
# ============================================================

# --- Ingest triples with confidence accumulation ---
loaded = 0
skipped = 0
accumulated = 0

with driver.session() as session:
    for t in triples:
        source_id = t.get("source_id", "UNKNOWN")
        target = t.get("target_citation")
        action = t.get("action", "RELATES_TO")
        confidence = t.get("confidence", 1.0)  # Default 1.0 for legacy triples

        # Skip if target is null
        if not target:
            skipped += 1
            continue

        # Determine source type from the ID
        if any(prefix in source_id for prefix in ["uksc_", "ewca_", "ewhc_", "ukut_"]):
            source_type = "CaseLaw"
        else:
            source_type = "Legislation"

        # Cypher: create nodes and relationship with confidence accumulation
        # MERGE ON CREATE/ON MATCH handles dedup + confidence accumulation
        # using MedKGent formula: s = 1 - (1-s)(1-s')
        query = """
        MERGE (s:LegalDoc {id: $source_id})
        SET s.type = $source_type,
            s.title = COALESCE(s.title, $source_title)
        MERGE (t:LegalDoc {citation: $target})
        SET t.act_name = COALESCE(t.act_name, $target_act)

        MERGE (s)-[r:LEGAL_RELATIONSHIP {action_type: $action}]->(t)
        ON CREATE SET
            r.detail = $detail,
            r.date = $date,
            r.confidence = $confidence,
            r.source_ids = [$source_id],
            r.times_seen = 1
        ON MATCH SET
            r.confidence = 1.0 - (1.0 - r.confidence) * (1.0 - $confidence),
            r.source_ids = CASE
                WHEN NOT $source_id IN COALESCE(r.source_ids, [])
                THEN COALESCE(r.source_ids, []) + $source_id
                ELSE r.source_ids
            END,
            r.times_seen = COALESCE(r.times_seen, 1) + 1,
            r.detail = COALESCE($detail, r.detail),
            r.date = COALESCE($date, r.date)
        """
        try:
            session.run(query,
                source_id=source_id,
                source_type=source_type,
                source_title=t.get("source_title", source_id),
                target=target,
                target_act=t.get("target_act_name", target),
                action=action,
                detail=t.get("detail_text"),
                date=t.get("effective_date"),
                confidence=confidence,
            )
        except Exception as e:
            print(f"   ❌ Error on triple {loaded}: {e}")
            skipped += 1
            continue

        loaded += 1

        if loaded % 200 == 0:
            print(f"   ... loaded {loaded} triples")

print(f"\n{'='*50}")
print(f"✅ NEO4J INGESTION COMPLETE")
print(f"   Loaded: {loaded} triples")
print(f"   Skipped (null targets/errors): {skipped}")

# --- Verify: print graph stats ---
with driver.session() as session:
    nodes = session.run("MATCH (n:LegalDoc) RETURN count(n) as cnt").single()["cnt"]
    edges = session.run("MATCH ()-[r]->() RETURN count(r) as cnt").single()["cnt"]
    actions = session.run("""
        MATCH ()-[r:LEGAL_RELATIONSHIP]->()
        RETURN r.action_type AS action, count(*) AS count
        ORDER BY count DESC
    """)
    action_counts = {r["action"]: r["count"] for r in actions}

    # Check confidence stats (MedKGent-style)
    conf_stats = session.run("""
        MATCH ()-[r:LEGAL_RELATIONSHIP]->()
        WHERE r.times_seen > 1
        RETURN count(r) AS accumulated_edges,
               avg(r.confidence) AS avg_confidence,
               max(r.times_seen) AS max_times_seen
    """).single()

print(f"\n📊 Graph Stats:")
print(f"   Nodes: {nodes}")
print(f"   Edges: {edges}")
print(f"   Actions: {action_counts}")

if conf_stats and conf_stats["accumulated_edges"]:
    print(f"\n📈 Confidence Accumulation Stats:")
    print(f"   Edges seen >1 time: {conf_stats['accumulated_edges']}")
    print(f"   Avg confidence (accumulated): {conf_stats['avg_confidence']:.4f}")
    print(f"   Max times seen: {conf_stats['max_times_seen']}")

driver.close()
print("\n✅ Neo4j connection closed")


✅ Connected to Neo4j at bolt://localhost:7687
📂 Loaded 8731 triples from data/extracted_triples.json
🗑️  Cleared existing graph
📇 Created indexes on LegalDoc.id and LegalDoc.citation
   ... loaded 200 triples
   ... loaded 400 triples
   ... loaded 600 triples
   ... loaded 800 triples
   ... loaded 1000 triples
   ... loaded 1200 triples
   ... loaded 1400 triples
   ... loaded 1600 triples
   ... loaded 1800 triples
   ... loaded 2000 triples
   ... loaded 2200 triples
   ... loaded 2400 triples
   ... loaded 2600 triples
   ... loaded 2800 triples
   ... loaded 3000 triples
   ... loaded 3200 triples
   ... loaded 3400 triples
   ... loaded 3600 triples
   ... loaded 3800 triples
   ... loaded 4000 triples
   ... loaded 4200 triples
   ... loaded 4400 triples
   ... loaded 4600 triples
   ... loaded 4800 triples
   ... loaded 5000 triples
   ... loaded 5200 triples
   ... loaded 5400 triples
   ... loaded 5600 triples
   ... loaded 5800 triples
   ... loaded 6000 triples
   ... load

In [40]:
################################################################
# CELL 14 — Interactive Visualization (PyVis)
################################################################

# !pip install pyvis networkx -q

from pyvis.network import Network

with open(OUTPUT_FILE, "r") as f:
    triples = json.load(f)

ACTION_COLORS = {
    "REPEALS": "#e74c3c",
    "AMENDS": "#f39c12",
    "SUBSTITUTES": "#9b59b6",
    "INSERTS": "#2ecc71",
    "COMMENCES": "#3498db",
    "REVOKES": "#e67e22",
    "OVERRULES": "#c0392b",
    "APPLIES": "#1abc9c",
    "CITES": "#95a5a6",
}

net = Network(
    height="700px", width="100%",
    bgcolor="#1a1a2e", font_color="#ffffff",
    directed=True, notebook=True, cdn_resources="remote"
)
net.barnes_hut(gravity=-3000, central_gravity=0.3, spring_length=200, spring_strength=0.05)

source_nodes = set()
target_nodes = set()
for t in triples:
    source_prefix = t["source_id"].rsplit('.xml_', 1)[0] if '.xml_' in t["source_id"] else t["source_id"]
    source_nodes.add((source_prefix, t.get("source_title", source_prefix)))
    if t.get("target_citation"):
        target_nodes.add(t["target_citation"])

for node_id, title in source_nodes:
    label = title[:40] + "..." if len(title) > 40 else title
    net.add_node(node_id, label=label, color="#3498db", size=20, shape="dot", title=f"Source: {title}")

for node in target_nodes:
    act = extract_act_name(node) or node
    label = act[:40] + "..." if len(act) > 40 else act
    net.add_node(node, label=label, color="#f1c40f", size=15, shape="dot", title=f"Target: {node}")

for t in triples:
    if not t.get("target_citation"):
        continue
    source_prefix = t["source_id"].rsplit('.xml_', 1)[0] if '.xml_' in t["source_id"] else t["source_id"]
    color = ACTION_COLORS.get(t["action"], "#ffffff")
    tooltip = t["action"]
    if t.get("effective_date"):
        tooltip += f" | {t['effective_date']}"
    net.add_edge(source_prefix, t["target_citation"], label=t["action"], color=color, width=2, title=tooltip, arrows="to")

net.show("kg_visualization.html")
print(f"✅ Visualization saved: {len(source_nodes)} sources, {len(target_nodes)} targets, {len(triples)} edges")


kg_visualization.html
✅ Visualization saved: 39 sources, 5715 targets, 8731 edges


In [43]:
"""
LegalKGent — Graph RAG Query Agent (Mistral)
=============================================
ReAct agent with run_cypher tool for querying the Legal Knowledge Graph.

Improvements (MedKGent-inspired):
  - Hallucination guardrail: agent must run at least one query before answering
  - First-ACTION parsing: only executes the first ACTION per LLM turn
  - Confidence-aware queries: can filter by r.confidence
  - Graph-context aware prompting

Copy each CELL block into a separate Jupyter notebook cell.
"""

# ============================================================
# CELL 1: Install dependencies
# ============================================================
# !pip install mistralai neo4j

# ============================================================
# CELL 2: Imports & Config
# ============================================================
import json
import os
from neo4j import GraphDatabase
from mistralai import Mistral

# --- CONFIG ---
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "LegalPassword123"

MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY", "wTPmklMivGd5WB14RsLnms5DUw1pOeHh")
MISTRAL_MODEL = "mistral-large-latest"

# Initialize clients
client = Mistral(api_key=MISTRAL_API_KEY)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("✅ Connected to Neo4j and Mistral")


# ============================================================
# CELL 3: Tool — run_cypher
# ============================================================
def run_cypher(query: str) -> str:
    """Execute a Cypher query against Neo4j and return results as JSON."""
    print(f"\n🔧 [TOOL CALL] run_cypher")
    print(f"   Query: {query}")
    with driver.session() as session:
        try:
            result = session.run(query)
            records = [dict(r) for r in result]
            output = json.dumps(records, indent=2, default=str)
            print(f"   ✅ Returned {len(records)} record(s)")
            # Print truncated output for readability
            preview = output[:2000]
            print(f"   Data: {preview}{'...' if len(output) > 2000 else ''}")
            return output
        except Exception as e:
            err = f"CYPHER ERROR: {e}"
            print(f"   ❌ {err}")
            return err


# ============================================================
# CELL 4: Build Schema Context
# ============================================================
def get_schema() -> dict:
    """Fetch graph schema for the system prompt."""
    with driver.session() as s:
        node_count = s.run("MATCH (n) RETURN count(n) AS cnt").single()["cnt"]
        edge_count = s.run("MATCH ()-[r]->() RETURN count(r) AS cnt").single()["cnt"]

        # Action type distribution
        actions = {r["action"]: r["count"] for r in s.run(
            "MATCH ()-[r:LEGAL_RELATIONSHIP]->() "
            "RETURN r.action_type AS action, count(*) AS count "
            "ORDER BY count DESC"
        )}

        # Sample edges (show source_id, action, target_citation, detail)
        samples = [dict(r) for r in s.run(
            "MATCH (s)-[r:LEGAL_RELATIONSHIP]->(t) "
            "RETURN s.id AS source_id, r.action_type AS action, "
            "r.detail AS detail, t.citation AS target "
            "LIMIT 8"
        )]

        # Get sample source IDs from the graph
        source_ids = [dict(r) for r in s.run("""
            MATCH (s:LegalDoc)-[:LEGAL_RELATIONSHIP]->()
            WHERE s.id IS NOT NULL
            RETURN DISTINCT s.id AS source_id
            ORDER BY s.id LIMIT 30
        """)]

        # Top target acts (most referenced)
        top_targets = [dict(r) for r in s.run("""
            MATCH ()-[r:LEGAL_RELATIONSHIP]->(t:LegalDoc)
            WHERE t.citation IS NOT NULL
            RETURN t.citation AS target, r.action_type AS action, count(*) AS cnt
            ORDER BY cnt DESC LIMIT 15
        """)]

    return {
        "node_count": node_count,
        "edge_count": edge_count,
        "action_types": actions,
        "sample_edges": samples,
        "source_ids": source_ids,
        "top_targets": top_targets,
    }

schema = get_schema()
print(f"📊 Graph: {schema['node_count']} nodes, {schema['edge_count']} edges")
print(f"   Actions: {schema['action_types']}")


# ============================================================
# CELL 5: Build Source ID → Title Lookup (from triples file)
# ============================================================

# Build a lookup from the triples file so the LLM can map Act names to IDs
SOURCE_LOOKUP = {}
TRIPLES_FILE = "data/extracted_triples.json"

if os.path.exists(TRIPLES_FILE):
    with open(TRIPLES_FILE, "r") as f:
        _triples = json.load(f)
    for t in _triples:
        sid = t.get("source_id", "")
        title = t.get("source_title", "")
        if sid and title:
            prefix = sid.rsplit(".xml_", 1)[0] if ".xml_" in sid else sid
            if prefix not in SOURCE_LOOKUP:
                SOURCE_LOOKUP[prefix] = title
    del _triples  # free memory
    print(f"📋 Source lookup: {len(SOURCE_LOOKUP)} Acts")

# Format for the prompt
SOURCE_TABLE = "\n".join(
    f"  {prefix} = {title}"
    for prefix, title in sorted(SOURCE_LOOKUP.items())
)


# ============================================================
# CELL 6: System Prompt (Fine-Tuned for this Graph)
# ============================================================

SYSTEM_PROMPT = f"""You are a UK Legal Knowledge Graph agent. You answer legal questions by querying a Neo4j database containing UK legislation relationships.

## ⚠️ CRITICAL GRAPH RULES (read these FIRST)

1. ALL relationships are `:LEGAL_RELATIONSHIP` — there are NO typed edges like :AMENDS or :REPEALS
2. The action is stored as a PROPERTY: `r.action_type` (values: AMENDS, REPEALS, SUBSTITUTES, INSERTS, COMMENCES, REVOKES, APPLIES)
3. Source nodes have `s.id` (e.g., "ukpga_2023_36.xml_1"). The ID includes ".xml_" + section number
4. Target nodes have `t.citation` (e.g., "Housing and Regeneration Act 2008 s.107")
5. NEVER guess document IDs — use the lookup table below or search with CONTAINS

## Source Document ID → Act Title Lookup
{SOURCE_TABLE}

## How to Search

### Find what an Act DOES (outgoing):
```
MATCH (s)-[r:LEGAL_RELATIONSHIP]->(t)
WHERE s.id STARTS WITH 'ukpga_2023_36'
RETURN s.id, r.action_type, r.detail, t.citation
LIMIT 25
```

### Find what AFFECTS an Act (incoming):
```
MATCH (s)-[r:LEGAL_RELATIONSHIP]->(t)
WHERE t.citation CONTAINS 'Housing and Regeneration Act 2008'
RETURN s.id, r.action_type, r.detail, t.citation
LIMIT 25
```

### Filter by action type:
```
MATCH (s)-[r:LEGAL_RELATIONSHIP]->(t)
WHERE s.id STARTS WITH 'ukpga_2023_36' AND r.action_type = 'REPEALS'
RETURN s.id, r.action_type, t.citation
```

### Count by action type:
```
MATCH (s)-[r:LEGAL_RELATIONSHIP]->(t)
WHERE s.id STARTS WITH 'ukpga_2023_36'
RETURN r.action_type AS action, count(*) AS count ORDER BY count DESC
```

### Find most amended/referenced targets:
```
MATCH (s)-[r:LEGAL_RELATIONSHIP]->(t)
RETURN t.citation, r.action_type, count(*) AS cnt ORDER BY cnt DESC LIMIT 15
```

## ⚠️ GROUNDING RULES — READ CAREFULLY

1. You MUST run at least one Cypher query BEFORE answering
2. Your answer MUST be based ONLY on the data returned by your queries
3. If a query returns 0 results, say so — do NOT fill in from general knowledge
4. If the graph does not contain information about a topic, say "The knowledge graph does not contain information about [topic]"
5. NEVER fabricate section numbers, Act names, or legal provisions not found in query results

## Graph Stats
- {{schema['node_count']}} nodes, {{schema['edge_count']}} edges
- Action distribution: {{json.dumps(schema['action_types'])}}

## Sample Edges
{json.dumps(schema['sample_edges'][:5], indent=2, default=str)}

## Your Tool
You have ONE tool: `run_cypher(query)` — executes Cypher and returns JSON results.

## Response Format (STRICT)

Every response MUST contain EXACTLY ONE of these:

THOUGHT: <your reasoning>
ACTION: run_cypher("<your Cypher query>")

OR (only after at least one successful query):

THOUGHT: <final interpretation of query results>
ANSWER: <your answer citing ONLY data from query results>

IMPORTANT: Output ONLY ONE ACTION per response. Wait for the result before deciding next steps.

## Rules
1. ALWAYS run at least one Cypher query before answering
2. Use STARTS WITH on s.id for source Act searches (e.g., s.id STARTS WITH 'ukpga_2023_36')
3. Use CONTAINS on t.citation for target Act searches
4. If you get 0 results, try the OTHER side (source vs target)
5. LIMIT to 25 unless the user asks for all
6. Include r.detail in your RETURN when available — it contains the exact amendment wording
7. Cite specific sections in your answer (e.g., "Section 3 of the Finance Act 2023 amends...")
8. Output only ONE ACTION per turn — do NOT chain multiple ACTIONs
"""


# ============================================================
# CELL 7: ReAct Agent Loop
# ============================================================

def agent_ask(question: str, max_steps: int = 7) -> str:
    """
    ReAct agent loop with run_cypher tool.
    Uses Mistral for reasoning.

    Improvements over baseline:
      - Parses FIRST ACTION only (not last) — proper ReAct one-action-per-turn
      - Hallucination guardrail: blocks ANSWER on step 1 (before any query)
      - Grounding check: warns if answering without non-empty results
    """
    print(f"\n{'#'*60}")
    print(f"  🔍 USER QUESTION: {question}")
    print(f"{'#'*60}")

    conversation = question
    has_nonempty_result = False  # Track whether we got any actual data

    for step in range(1, max_steps + 1):
        print(f"\n--- Agent Step {step} ---")
        print(f"📤 [LLM REQUEST] Sending to Mistral ({MISTRAL_MODEL})...")

        response = client.chat.complete(
            model=MISTRAL_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": conversation}
            ],
            temperature=0.1,
        )

        llm_output = response.choices[0].message.content.strip()
        print(f"📥 [LLM RESPONSE]\n{llm_output}")

        # --- Parse ANSWER ---
        if "ANSWER:" in llm_output:
            # Hallucination guardrail: agent must run at least one query before answering
            if step == 1:
                print("⚠️  Blocked ANSWER on step 1 — agent must query the graph first")
                conversation += (
                    f"\n\n{llm_output}\n\n"
                    f"You MUST run at least one Cypher query before answering. "
                    f"Do NOT answer from general knowledge. Use ACTION: run_cypher(\"...\") first."
                )
                continue

            answer = llm_output.split("ANSWER:")[-1].strip()

            # Grounding warning
            if not has_nonempty_result:
                print("⚠️  WARNING: Agent answered without receiving any non-empty query results")
                answer = (
                    "⚠️ Note: The knowledge graph did not return relevant data for this query. "
                    "The following answer may not be fully grounded.\n\n" + answer
                )

            print(f"\n{'='*60}")
            print(f"  ✅ FINAL ANSWER")
            print(f"{'='*60}")
            print(answer)
            return answer

        # --- Parse ACTION (take FIRST action only — one action per ReAct turn) ---
        if "ACTION:" in llm_output:
            action_line = llm_output.split("ACTION:")[1].strip()

            # If there are more ACTION: lines after this one, truncate
            if "ACTION:" in action_line:
                action_line = action_line.split("ACTION:")[0].strip()

            tool_result = None

            if "run_cypher(" in action_line:
                # Extract Cypher query (handle nested parentheses)
                start = action_line.index("run_cypher(") + len("run_cypher(")
                depth = 1
                end = start
                for i in range(start, len(action_line)):
                    if action_line[i] == '(':
                        depth += 1
                    elif action_line[i] == ')':
                        depth -= 1
                    if depth == 0:
                        end = i
                        break
                cypher = action_line[start:end].strip().strip("\"'`")
                tool_result = run_cypher(cypher)

                # Track whether we got non-empty results
                if tool_result and tool_result.strip() not in ('[]', '""', 'null'):
                    try:
                        parsed = json.loads(tool_result)
                        if isinstance(parsed, list) and len(parsed) > 0:
                            has_nonempty_result = True
                    except json.JSONDecodeError:
                        pass

            if tool_result is not None:
                conversation += (
                    f"\n\n{llm_output}"
                    f"\n\nOBSERVATION (tool result):\n{tool_result}"
                    f"\n\nAnalyze the results. If you have enough information, respond with ANSWER:. "
                    f"Otherwise, run another ACTION: run_cypher(...)."
                )
            else:
                print(f"⚠️  Could not parse action: {action_line}")
                conversation += (
                    f"\n\n{llm_output}"
                    f"\n\nOBSERVATION: Could not parse that action. "
                    f"Use this format: ACTION: run_cypher(\"MATCH (s)-[r:LEGAL_RELATIONSHIP]->(t) WHERE ... RETURN ...\")"
                )
        else:
            print("⚠️  No ACTION or ANSWER found, nudging agent...")
            conversation += (
                f"\n\n{llm_output}\n\n"
                f"You must respond with ACTION: run_cypher(\"...\") or ANSWER: <your answer>."
            )

    return "❌ Agent did not reach an answer within max steps."


# ============================================================
# CELL 8: Test Questions
# ============================================================

# Lawyer-relevant test questions grounded in the actual graph data:
#
# --- Simple (single-Act lookups) ---
# result = agent_ask("What does the Social Housing (Regulation) Act 2023 amend in the Housing and Regeneration Act 2008?")
# result = agent_ask("Has the Social Housing Act 2023 repealed any parts of the Housing and Regeneration Act 2008?")
#
# --- Multi-source (cross-Act aggregation — KG strength) ---
# result = agent_ask("Which 2023 Acts modify the Employment Rights Act 1996, and what type of changes does each make?")
#
# --- Multi-hop chain (graph traversal — KG strength) ---
# result = agent_ask("Trace the chain of modifications to the Energy Profits Levy: which Act changed the rate, and until when has the levy been extended?")
#
# --- Detail extraction ---
# result = agent_ask("What is the exact wording substituted into section 107 of the Housing and Regeneration Act 2008?")
#
# --- Finance / tax ---
# result = agent_ask("What changes did the Finance Act 2023 make to the dividend allowance under the Income Tax Act 2007?")
# result = agent_ask("What is the current annual exempt amount for Capital Gains Tax, and which legislation reduced it?")
#
# --- Conflict detection ---
# result = agent_ask("Are there any sections that are both amended and repealed by different Acts?")
#
# --- Aggregation ---
# result = agent_ask("Which Act has the highest number of legal relationships overall? Break down by type.")

# result = agent_ask("Which 2023 Acts modify the Employment Rights Act 1996, and what type of changes does each make?")


# ============================================================
# CELL 9: Cleanup
# ============================================================
# driver.close()
# print("✅ Neo4j connection closed")


✅ Connected to Neo4j and Mistral
📊 Graph: 7950 nodes, 8731 edges
   Actions: {'SUBSTITUTES': 1467, 'INSERTS': 1450, 'AMENDS': 1138, 'REQUIRES': 1070, 'DEFINES': 991, 'REPEALS': 883, 'EMPOWERS': 556, 'APPLIES': 456, 'CREATES': 185, 'INTERPRETS': 143, 'PROHIBITS': 114, 'DELEGATES': 108, 'COMMENCES': 96, 'CITES': 26, 'OVERRULES': 24, 'EXTENDS': 14, 'REVOKES': 6, 'IMPLEMENTS': 4}
📋 Source lookup: 39 Acts

############################################################
  🔍 USER QUESTION: Which 2023 Acts modify the Employment Rights Act 1996, and what type of changes does each make?
############################################################

--- Agent Step 1 ---
📤 [LLM REQUEST] Sending to Mistral (mistral-large-latest)...
📥 [LLM RESPONSE]
THOUGHT: To find which 2023 Acts modify the Employment Rights Act 1996, I need to search for relationships where the target citation contains "Employment Rights Act 1996" and the source is a 2023 Act. I'll filter for source IDs starting with "ukpga_2023" to 

In [44]:
#!/usr/bin/env python3
"""
LegalKGent — Corpus Validator
==============================
Validates downloaded data for quality, completeness, and stats.
Run after download_transport_data.py.

Usage:
    python validate_corpus.py
"""

import os
import json
import statistics
import xml.etree.ElementTree as ET
from collections import Counter


# ============================================================
# CONFIG
# ============================================================

DATA_DIR = "data"
LEGISLATION_DIR = os.path.join(DATA_DIR, "raw_legislation")
CASELAW_DIR = os.path.join(DATA_DIR, "raw_caselaw")
SI_DIR = os.path.join(DATA_DIR, "raw_statutory_instruments")
NOTES_DIR = os.path.join(DATA_DIR, "explanatory_notes")
AMENDMENTS_DIR = os.path.join(DATA_DIR, "amendments")

# Namespace helpers
def strip_ns(tag):
    """Strip XML namespace prefix."""
    if '}' in tag:
        return tag.split('}', 1)[1]
    return tag


# ============================================================
# VALIDATE XML FILES
# ============================================================

def validate_xml_dir(dirpath, label):
    """Validate all XML files in a directory."""
    print(f"\n{'='*50}")
    print(f"📂 {label}: {dirpath}")
    print(f"{'='*50}")
    
    if not os.path.exists(dirpath):
        print(f"  [WARN] Directory does not exist")
        return {}
    
    files = [f for f in os.listdir(dirpath) if f.endswith('.xml')]
    if not files:
        print(f"  [WARN] No XML files found")
        return {}
    
    stats = {
        'total_files': len(files),
        'valid_xml': 0,
        'parse_errors': 0,
        'total_sections': 0,
        'total_size_kb': 0,
        'file_sizes': [],
        'titles': [],
        'errors': [],
    }
    
    for filename in sorted(files):
        filepath = os.path.join(dirpath, filename)
        file_size = os.path.getsize(filepath) / 1024
        stats['total_size_kb'] += file_size
        stats['file_sizes'].append(file_size)
        
        try:
            tree = ET.parse(filepath)
            root = tree.getroot()
            stats['valid_xml'] += 1
            
            # Count sections (P1 for legislation, paragraph for caselaw)
            section_count = 0
            title = "Unknown"
            
            for elem in root.iter():
                tag = strip_ns(elem.tag)
                if tag == 'P1':
                    section_count += 1
                elif tag == 'paragraph':
                    section_count += 1
                elif tag == 'title' and title == "Unknown":
                    if elem.text:
                        title = elem.text.strip()[:80]
                elif tag == 'FRBRname':
                    title = elem.get('value', title)[:80]
            
            stats['total_sections'] += section_count
            stats['titles'].append(f"  {filename}: {title} ({section_count} sections, {file_size:.0f} KB)")
            
        except ET.ParseError as e:
            stats['parse_errors'] += 1
            stats['errors'].append(f"  [{filename}] XML Parse Error: {e}")
    
    # Print report
    print(f"\n  Files: {stats['total_files']}")
    print(f"  Valid XML: {stats['valid_xml']}")
    if stats['parse_errors']:
        print(f"  ❌ Parse Errors: {stats['parse_errors']}")
    print(f"  Total Sections: {stats['total_sections']}")
    print(f"  Total Size: {stats['total_size_kb']:.0f} KB ({stats['total_size_kb']/1024:.1f} MB)")
    
    if stats['file_sizes']:
        print(f"\n  File Size Stats:")
        print(f"    Min: {min(stats['file_sizes']):.0f} KB")
        print(f"    Max: {max(stats['file_sizes']):.0f} KB")
        print(f"    Avg: {statistics.mean(stats['file_sizes']):.0f} KB")
    
    print(f"\n  Documents:")
    for t in stats['titles'][:20]:
        print(t)
    if len(stats['titles']) > 20:
        print(f"  ... and {len(stats['titles'])-20} more")
    
    if stats['errors']:
        print(f"\n  Errors:")
        for e in stats['errors']:
            print(e)
    
    return stats


# ============================================================
# VALIDATE PARSED CORPUS (if exists)
# ============================================================

def validate_parsed_corpus(filepath):
    """Validate a parsed JSON corpus file."""
    print(f"\n{'='*50}")
    print(f"📊 PARSED CORPUS: {filepath}")
    print(f"{'='*50}")
    
    if not os.path.exists(filepath):
        print(f"  [INFO] Not yet created (run kg_creation.py first)")
        return
    
    with open(filepath, "r") as f:
        data = json.load(f)
    
    total = len(data)
    print(f"\n  Total Chunks: {total}")
    
    # Source distribution
    sources = Counter(item.get('source', 'unknown') for item in data)
    print(f"\n  Source Distribution:")
    for src, count in sources.most_common():
        print(f"    {src}: {count} ({count/total:.1%})")
    
    # Content length stats
    lengths = [len(item.get('content', '')) for item in data]
    if lengths:
        print(f"\n  Content Length (chars):")
        print(f"    Min: {min(lengths)}")
        print(f"    Max: {max(lengths)}")
        print(f"    Avg: {int(statistics.mean(lengths))}")
        print(f"    Median: {int(statistics.median(lengths))}")
    
    # Empty content check
    empty = sum(1 for l in lengths if l < 10)
    if empty:
        print(f"\n  ⚠️  Empty chunks (<10 chars): {empty}")
    
    # Missing fields
    required = ['id', 'source', 'content', 'doc_title']
    for field in required:
        missing = sum(1 for item in data if not item.get(field))
        if missing:
            print(f"  ⚠️  Missing '{field}': {missing}")
    
    # Chunk budget check
    print(f"\n  📏 Chunk Budget:")
    print(f"    Current: {total}")
    print(f"    Limit: 5,000")
    print(f"    Remaining: {5000-total}")
    if total > 5000:
        print(f"    ❌ OVER BUDGET by {total-5000} chunks!")
    else:
        print(f"    ✅ Within budget")


# ============================================================
# MAIN
# ============================================================

def main():
    print(f"""
╔══════════════════════════════════════════════════╗
║  LegalKGent — Data Validation Report            ║
╚══════════════════════════════════════════════════╝
    """)
    
    all_stats = {}
    
    # Validate each data source
    all_stats['legislation'] = validate_xml_dir(LEGISLATION_DIR, "PRIMARY LEGISLATION")
    all_stats['caselaw'] = validate_xml_dir(CASELAW_DIR, "CASE LAW")
    all_stats['sis'] = validate_xml_dir(SI_DIR, "STATUTORY INSTRUMENTS")
    all_stats['notes'] = validate_xml_dir(NOTES_DIR, "EXPLANATORY NOTES")
    all_stats['amendments'] = validate_xml_dir(AMENDMENTS_DIR, "AMENDMENTS TABLES")
    
    # Validate parsed corpus if exists
    for corpus_file in ["data/smart_corpus.json", "data/legal_corpus_clean.json"]:
        validate_parsed_corpus(corpus_file)
    
    # Summary
    print(f"\n{'='*50}")
    print(f"📊 OVERALL SUMMARY")
    print(f"{'='*50}")
    
    total_files = sum(s.get('total_files', 0) for s in all_stats.values())
    total_sections = sum(s.get('total_sections', 0) for s in all_stats.values())
    total_size = sum(s.get('total_size_kb', 0) for s in all_stats.values())
    
    print(f"  Total XML files: {total_files}")
    print(f"  Total sections: {total_sections}")
    print(f"  Total disk size: {total_size/1024:.1f} MB")
    print(f"  Est. chunks (sections): ~{total_sections}")
    print(f"  Budget remaining: ~{5000-total_sections} chunks")


if __name__ == "__main__":
    main()



╔══════════════════════════════════════════════════╗
║  LegalKGent — Data Validation Report            ║
╚══════════════════════════════════════════════════╝
    

📂 PRIMARY LEGISLATION: data/raw_legislation

  Files: 17
  Valid XML: 17
  Total Sections: 5286
  Total Size: 35315 KB (34.5 MB)

  File Size Stats:
    Min: 8 KB
    Max: 6950 KB
    Avg: 2077 KB

  Documents:
  ukpga_1980_34.xml: Transport Act 1980 (65 sections, 324 KB)
  ukpga_1984_27.xml: Road Traffic Regulation Act 1984 (372 sections, 3410 KB)
  ukpga_1988_52.xml: Road Traffic Act 1988 (327 sections, 4443 KB)
  ukpga_1988_53.xml: Road Traffic Offenders Act 1988 (140 sections, 2962 KB)
  ukpga_1993_43.xml: Railways Act 1993 (371 sections, 4026 KB)
  ukpga_2000_38.xml: Transport Act 2000 (1189 sections, 6950 KB)
  ukpga_2005_14.xml: Railways Act 2005 (271 sections, 1828 KB)
  ukpga_2006_49.xml: Road Safety Act 2006 (321 sections, 1057 KB)
  ukpga_2008_26.xml: Local Transport Act 2008 (347 sections, 1499 KB)
  ukpga_2012_

In [45]:
qss="Assume I am advising a manufacturer of autonomous vehicles. Based on the statutory network in your graph, how do the provisions of the Automated and Electric Vehicles Act 2018 interact with the older framework of the Road Traffic Act 1988, and specifically, what operational or technical requirements are introduced by the Road Vehicles (Construction and Use) (Automated Vehicles) Order 2022 (SI 2022/470)?"
result = agent_ask(qss)



############################################################
  🔍 USER QUESTION: Assume I am advising a manufacturer of autonomous vehicles. Based on the statutory network in your graph, how do the provisions of the Automated and Electric Vehicles Act 2018 interact with the older framework of the Road Traffic Act 1988, and specifically, what operational or technical requirements are introduced by the Road Vehicles (Construction and Use) (Automated Vehicles) Order 2022 (SI 2022/470)?
############################################################

--- Agent Step 1 ---
📤 [LLM REQUEST] Sending to Mistral (mistral-large-latest)...
📥 [LLM RESPONSE]
THOUGHT: To answer this question, I need to:
1. Identify how the Automated and Electric Vehicles Act 2018 (AEVA 2018) interacts with the Road Traffic Act 1988 (RTA 1988) by querying for relationships between these two Acts.
2. Specifically look for amendments, substitutions, or other legal relationships introduced by AEVA 2018 that affect RTA 1988.
